# Quantum Pump BO Experiment - Hardware v5b

## Five-Phase Integrated Workflow (Dynamic Search Regions)

### Rule No.1
- Any voltage change must be **step-wise** with maximum step `< 5 mV`
- In this notebook, we enforce `MAX_VOLTAGE_STEP_V = 4 mV`

### Dynamic Search Strategy
- Phase 2 scan window is centered on Phase 1 pinch-off results
- Phase 4 BO bounds are constrained to Phase 2 valid region (`0 < I < 4 nA`)
- `V_p` search bounds are also hyperparameterized

### Phase 1
- Find pinch-off voltages for `V_ENT`, `V_P`, `V_EXIT` at gain `1e-7 A/V`

### Phase 2
- Find region where `0 < I < 4 nA` using 2D scan (`V_ENT`, `V_EXIT`) with fixed `V_P`

### Phase 3
- Experimenter reviews Phase 2 result
- `y`: manually switch amplifier dial to `1e-9 A/V`, then continue
- `n`: stop and save summary
- `r`: re-try Phase 2

### Phase 4
- Find `n = 1` region using constrained BO in dynamic bounds

### Phase 5
- Experimenter reviews Phase 4 result
- If `yes`, run pump-map plotting (6-panel)

In [ ]:
# Cell 1: Imports
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LogNorm
from pathlib import Path
from datetime import datetime
import warnings
import pandas as pd
import time
warnings.filterwarnings('ignore')

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from scipy.stats import norm, qmc
from scipy.optimize import minimize
from scipy.ndimage import label
from scipy.interpolate import RegularGridInterpolator, LinearNDInterpolator, NearestNDInterpolator

# Hardware control
try:
    import pyvisa
    PYVISA_AVAILABLE = True
    print('✅ PyVISA imported successfully')
except ImportError:
    PYVISA_AVAILABLE = False
    print('⚠️ PyVISA not available - Simulation mode only')

print('='*60)
print('Quantum Pump BO Experiment - Hardware v5b')
print('Five-Phase Workflow (v5b): Dynamic bounds from phase outcomes')
print('='*60)

In [ ]:
# Cell 2: Configuration

class Config:
    """
    Experiment Configuration for 5-Phase Workflow (v5b)
    """

    # ==================== PHYSICAL CONSTANTS ====================
    e = 1.60217663e-19
    f = 6.5e7

    @property
    def target_current(self):
        return self.e * self.f

    # ==================== GAIN PROFILE ====================
    # Phase 1/2: 1e-7 A/V, Phase 4/5: 1e-9 A/V (after manual dial switch at Phase 3)
    GAIN_PHASE1_A_PER_V = 1e-7
    GAIN_PHASE2_A_PER_V = 1e-7
    GAIN_PHASE4_A_PER_V = 1e-9
    CURRENT_AMP_GAIN = GAIN_PHASE1_A_PER_V

    # ==================== RULE NO.1: VOLTAGE STEP ====================
    MAX_VOLTAGE_STEP_V = 0.004
    SETTLING_TIME = 0.08

    # ==================== GPIB ADDRESSES ====================
    ADDR_YOKO_ENT  = "GPIB43::1::INSTR"
    ADDR_YOKO_P    = "GPIB43::2::INSTR"
    ADDR_YOKO_EXIT = "GPIB43::8::INSTR"
    ADDR_DMM       = "GPIB43::19::INSTR"

    # ==================== PHASE 1: PINCH-OFF ====================
    PINCH_SCAN_RANGES = {
        'V_ent': (-0.3, 0.3),
        'V_p': (-0.3, 0.3),
        'V_exit': (-0.3, 0.3),
    }
    PINCH_SCAN_POINTS = 121
    PINCH_REF_V_ENT = 0.0
    PINCH_REF_V_P = 0.0
    PINCH_REF_V_EXIT = 0.0
    PINCH_OFF_THRESHOLD_A = 0.5e-9

    # ==================== PHASE 2: SPARSE ADAPTIVE 3D SEARCH ====================
    # Dynamic cube centered on Phase 1 pinch-off (with optional offsets)
    PHASE2_RANGE_V_ENT = 0.30   # total range: ±150 mV
    PHASE2_RANGE_V_P = 0.30     # total range: ±150 mV
    PHASE2_RANGE_V_EXIT = 0.30  # total range: ±150 mV

    # Optional fixed-V_P preset (keeps sparse/adaptive logic, collapses V_P span if enabled)
    USE_FIXED_V_P_MODE = False
    FIXED_V_P_VALUE = 0.2          # V
    FIXED_V_P_PHASE2_RANGE = 0.0   # V (0 => exact fixed value)

    PHASE2_CENTER_FROM_PHASE1_PINCH = True
    PHASE2_CENTER_OFFSET_V_ENT = 0.0
    PHASE2_CENTER_OFFSET_V_P = 0.0
    PHASE2_CENTER_OFFSET_V_EXIT = 0.0

    PHASE2_I_MIN_A = 0.0
    PHASE2_I_MAX_A = 4e-9

    # Sparse adaptive 3D sampling budget
    PHASE2_N_INITIAL_3D = 45
    PHASE2_N_ADAPTIVE_3D = 75
    PHASE2_N_CANDIDATES_3D = 3000
    PHASE2_RANDOM_SEED = 42

    # Adaptive score = P(feasible)*exp(-|n-1|/tau) + w*normalized_uncertainty
    PHASE2_SCORE_TAU_N = 0.08
    PHASE2_SCORE_UNCERT_WEIGHT = 0.20

    # ==================== PHASE 4: CONSTRAINED n=1 BO ====================
    PHASE4_TARGET_N = 1.0
    PHASE4_N_INITIAL = 20
    PHASE4_N_ITER = 80
    PHASE4_EARLY_STOP_PATIENCE = 25

    # V_p search hyperparameters (dynamic)
    PHASE4_RANGE_V_P = 0.12      # total V_p range around center
    PHASE4_VP_OFFSET = 0.0       # center shift from phase2 best point

    # Bounds strategy from Phase 2 valid region
    PHASE4_USE_PHASE2_VALID_BOUNDS = True
    PHASE4_MARGIN_V_ENT = 0.0
    PHASE4_MARGIN_V_P = 0.0
    PHASE4_MARGIN_V_EXIT = 0.0

    PHASE4_N_TOL = 0.03
    PHASE4_PLATEAU_TOP_K = 8
    PHASE4_DV_EXIT = 0.006
    PHASE4_FLATNESS_WEIGHT = 0.35

    # ==================== PHASE 5: PUMP MAP ====================
    MAPPING_RANGE_V_ENT = 0.2
    MAPPING_RANGE_V_EXIT = 0.40
    N_MAPPING_LHS = 30
    N_MAPPING_ADAPTIVE = 70
    MAPPING_GRID_RESOLUTION = 80
    TRANSCONDUCTANCE_CLIP_PERCENTILE = 99.0
    CURVE_OFFSETS = [-0.04, -0.02, 0.02, 0.04]

    # Phase 5 bounds clipping + plateau-aware acquisition
    PHASE5_CLIP_TO_PHASE2_VALID_BOUNDS = True
    PHASE5_CLIP_TO_REPLAY_BOUNDS = True
    PHASE5_SUGGESTION_CANDIDATES = 2000

    PHASE5_ACQ_W_UNCERT = 0.45
    PHASE5_ACQ_W_LEVEL = 0.45
    PHASE5_ACQ_W_FLAT = 0.10
    PHASE5_ACQ_LEVELS = (1.0, 2.0)
    PHASE5_ACQ_LEVEL_TAU = 0.10
    PHASE5_ACQ_FLAT_DV = 0.004
    PHASE5_ACQ_FLAT_SLOPE_TAU = 12.0

    # Optional dense line refinement for better plateau curves
    PHASE5_ENABLE_DENSE_REFINEMENT = True
    PHASE5_REFINEMENT_N_LINES = 3
    PHASE5_REFINEMENT_TARGET_LEVELS = (1.0, 2.0)
    PHASE5_REFINEMENT_VEXIT_SPAN = 0.08
    PHASE5_REFINEMENT_STEP_V = 0.002

    # ==================== MODE ====================
    FORCE_SIMULATION = False

    # Simulation model: 'analytic' (default) or 'replay_2d' (data-driven pre-check)
    SIMULATION_MODEL_KIND = 'analytic'
    SIM_REPLAY_DATA_PATH = ''
    SIM_REPLAY_V_P_REF = 0.2
    SIM_REPLAY_V_P_SIGMA = 0.03
    SIM_REPLAY_NOISE_STD_N = 0.015

    # ==================== OUTPUT ====================
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = Path(f"./experiment_outputs_v5b_{timestamp}")

    @classmethod
    def print_settings(cls):
        cfg = cls()
        print('='*60)
        print('EXPERIMENT CONFIGURATION (v5b)')
        print('='*60)
        print(f'Frequency:                   {cls.f/1e9:.3f} GHz')
        print(f'Target current (n=1):        {cfg.target_current*1e12:.4f} pA (ef)')
        print(f'Max voltage step:            {cls.MAX_VOLTAGE_STEP_V*1e3:.1f} mV')

        print()
        print('--- Gain policy ---')
        print(f'Phase 1 gain:                {cls.GAIN_PHASE1_A_PER_V:.0e} A/V')
        print(f'Phase 2 gain:                {cls.GAIN_PHASE2_A_PER_V:.0e} A/V')
        print(f'Phase 4/5 gain:              {cls.GAIN_PHASE4_A_PER_V:.0e} A/V')

        print()
        print('--- Phase 1: Pinch-off ---')
        print(f'Scan points per gate:        {cls.PINCH_SCAN_POINTS}')
        print(f'Pinch-off threshold |I|:     {cls.PINCH_OFF_THRESHOLD_A*1e9:.3f} nA')

        print()
        print('--- Phase 2: Sparse adaptive 3D ---')
        print(f'Condition:                   {cls.PHASE2_I_MIN_A*1e9:.1f} < I < {cls.PHASE2_I_MAX_A*1e9:.1f} nA')
        print(f'Ranges (ENT/P/EXIT):         {cls.PHASE2_RANGE_V_ENT:.3f} / {cls.PHASE2_RANGE_V_P:.3f} / {cls.PHASE2_RANGE_V_EXIT:.3f} V')
        print(f'Fixed V_P mode:              {cls.USE_FIXED_V_P_MODE}')
        if cls.USE_FIXED_V_P_MODE:
            print(f'  Fixed V_P value:           {cls.FIXED_V_P_VALUE:+.4f} V')
            print(f'  Fixed V_P Phase2 range:    {cls.FIXED_V_P_PHASE2_RANGE:.4f} V')
        print(f'Center from pinch-off:       {cls.PHASE2_CENTER_FROM_PHASE1_PINCH}')
        print(f'Center offsets (ENT/P/EXIT): {cls.PHASE2_CENTER_OFFSET_V_ENT:+.3f} / {cls.PHASE2_CENTER_OFFSET_V_P:+.3f} / {cls.PHASE2_CENTER_OFFSET_V_EXIT:+.3f} V')
        print(f'Initial/Adaptive points:     {cls.PHASE2_N_INITIAL_3D} / {cls.PHASE2_N_ADAPTIVE_3D}')
        print(f'Candidates per step:         {cls.PHASE2_N_CANDIDATES_3D}')

        print()
        print('--- Phase 4: Dynamic constrained BO ---')
        print(f'Use phase2 valid bounds:     {cls.PHASE4_USE_PHASE2_VALID_BOUNDS}')
        print(f'Bounds margin (ENT/P/EXIT):  {cls.PHASE4_MARGIN_V_ENT:+.3f} / {cls.PHASE4_MARGIN_V_P:+.3f} / {cls.PHASE4_MARGIN_V_EXIT:+.3f} V')
        print(f'V_P total range:             {cls.PHASE4_RANGE_V_P:.3f} V')
        print(f'V_P center offset:           {cls.PHASE4_VP_OFFSET:+.3f} V')
        print(f'Initial points:              {cls.PHASE4_N_INITIAL}')
        print(f'Max BO iterations:           {cls.PHASE4_N_ITER}')
        print(f'Early stop patience:         {cls.PHASE4_EARLY_STOP_PATIENCE}')

        print()
        print('--- Phase 5: Pump map ---')
        print(f'LHS samples:                 {cls.N_MAPPING_LHS}')
        print(f'Adaptive samples:            {cls.N_MAPPING_ADAPTIVE}')
        print(f'Total mapping points:        {cls.N_MAPPING_LHS + cls.N_MAPPING_ADAPTIVE}')
        print(f'Clip to phase2 valid bounds: {cls.PHASE5_CLIP_TO_PHASE2_VALID_BOUNDS}')
        print(f'Clip to replay bounds:       {cls.PHASE5_CLIP_TO_REPLAY_BOUNDS}')
        print(f'Acq weights (u/level/flat):  {cls.PHASE5_ACQ_W_UNCERT:.2f} / {cls.PHASE5_ACQ_W_LEVEL:.2f} / {cls.PHASE5_ACQ_W_FLAT:.2f}')
        print(f'Acq target levels:           {cls.PHASE5_ACQ_LEVELS}')
        print(f'Dense refinement:            {cls.PHASE5_ENABLE_DENSE_REFINEMENT}')
        if cls.PHASE5_ENABLE_DENSE_REFINEMENT:
            print(f'  Lines/span/step:           {cls.PHASE5_REFINEMENT_N_LINES} / {cls.PHASE5_REFINEMENT_VEXIT_SPAN:.3f} V / {cls.PHASE5_REFINEMENT_STEP_V:.3f} V')

        print()
        print(f'Force simulation:            {cls.FORCE_SIMULATION}')
        print(f'Simulation model:            {cls.SIMULATION_MODEL_KIND}')
        if str(cls.SIMULATION_MODEL_KIND).lower() == 'replay_2d':
            print(f'  Replay data path:          {cls.SIM_REPLAY_DATA_PATH}')
            print(f'  Replay V_P ref/sigma:      {cls.SIM_REPLAY_V_P_REF:+.4f} / {cls.SIM_REPLAY_V_P_SIGMA:.4f} V')
            print(f'  Replay noise std (n):      {cls.SIM_REPLAY_NOISE_STD_N:.4f}')
        print('='*60)


Config.print_settings()

In [ ]:
# Cell 3: Instrument Controller

class InstrumentController:
    """
    Hardware interface with simulation fallback.

    Rule No.1 is enforced via set_voltages_stepwise:
    every voltage update is split so that each step <= MAX_VOLTAGE_STEP_V.
    """

    def __init__(self, config):
        self.cfg = config
        self.simulation_mode = False
        self.current_amp_gain = float(config.CURRENT_AMP_GAIN)
        self.current_voltages = {
            'V_ent': float(config.PINCH_REF_V_ENT),
            'V_p': float(config.PINCH_REF_V_P),
            'V_exit': float(config.PINCH_REF_V_EXIT),
        }

        if config.FORCE_SIMULATION:
            print('⚠️ FORCE_SIMULATION enabled')
            self.simulation_mode = True
            self._init_simulation()
            return

        if not PYVISA_AVAILABLE:
            print('⚠️ PyVISA not available')
            self.simulation_mode = True
            self._init_simulation()
            return

        try:
            self.rm = pyvisa.ResourceManager()
            print('Connecting to instruments...')

            self.yoko_ent = self.rm.open_resource(config.ADDR_YOKO_ENT)
            print(f'  ✅ G_ENT: {config.ADDR_YOKO_ENT}')

            self.yoko_p = self.rm.open_resource(config.ADDR_YOKO_P)
            print(f'  ✅ G_P: {config.ADDR_YOKO_P}')

            self.yoko_exit = self.rm.open_resource(config.ADDR_YOKO_EXIT)
            print(f'  ✅ G_EXIT: {config.ADDR_YOKO_EXIT}')

            self.dmm = self.rm.open_resource(config.ADDR_DMM)
            print(f'  ✅ DMM: {config.ADDR_DMM}')

            self._configure_instruments()
            print()
            print('✅ HARDWARE MODE')
            print(f'   Target current:      {config.target_current*1e12:.4f} pA (ef)')
            print(f'   Active amp gain:     {self.current_amp_gain:.0e} A/V')

        except Exception as e:
            print()
            print(f'⚠️ Hardware connection failed: {e}')
            print('   Using SIMULATION MODE')
            self.simulation_mode = True
            self._init_simulation()

    def _configure_instruments(self):
        for inst in [self.yoko_ent, self.yoko_p, self.yoko_exit]:
            inst.write_termination = "\n"
            inst.read_termination = "\n"
            inst.timeout = 5000
        self.dmm.write_termination = "\n"
        self.dmm.read_termination = "\n"
        self.dmm.timeout = 10000

    def _init_simulation(self):
        model_kind = str(getattr(self.cfg, 'SIMULATION_MODEL_KIND', 'analytic')).strip().lower()

        if model_kind in ['replay_2d', 'replay2d', 'measured_2d', 'realdata_2d']:
            if self._init_simulation_replay_2d():
                print('   Simulation model initialized: replay_2d')
                return
            print('   Replay model init failed. Falling back to analytic model.')

        self._init_simulation_analytic()
        print('   Simulation model initialized: analytic')

    def _init_simulation_analytic(self):
        self.simulation_model_kind = 'analytic'
        self.Va_base = -0.66
        self.Vb = 0.008
        self.delta2 = 0.10
        self.cross_coupling_ent = 0.30
        self.V_ent_ref = -0.62
        self.V_ent_center = -0.62
        self.ent_width = 0.20
        self.ent_sharpness = 100.0
        self.V_p_center = -0.63
        self.p_width = 0.20
        self.p_sharpness = 80.0
        self.noise_std = 1e-4

    def _init_simulation_replay_2d(self):
        raw_path = str(getattr(self.cfg, 'SIM_REPLAY_DATA_PATH', '')).strip()
        if raw_path == '':
            print('   SIM_REPLAY_DATA_PATH is empty')
            return False

        data_path = Path(raw_path).expanduser()
        if not data_path.is_absolute():
            data_path = Path.cwd() / data_path

        if not data_path.exists():
            print(f'   Replay data not found: {data_path}')
            return False

        try:
            arr = np.loadtxt(data_path)
        except Exception as e:
            print(f'   Failed to load replay data: {e}')
            return False

        if arr.ndim == 1:
            arr = arr.reshape(1, -1)
        if arr.shape[1] < 3:
            print('   Replay data must have at least 3 columns: V_ent, V_exit, I_nA')
            return False

        V_ent = arr[:, 0].astype(float)
        V_exit = arr[:, 1].astype(float)
        I_nA = arr[:, 2].astype(float)

        mask = np.isfinite(V_ent) & np.isfinite(V_exit) & np.isfinite(I_nA)
        V_ent = V_ent[mask]
        V_exit = V_exit[mask]
        I_nA = I_nA[mask]

        if len(V_ent) < 10:
            print('   Replay data has too few valid points')
            return False

        n_vals = (I_nA * 1e-9) / float(self.cfg.target_current)
        pts = np.column_stack([V_ent, V_exit])

        self.replay_interp_linear = LinearNDInterpolator(pts, n_vals, fill_value=np.nan)
        self.replay_interp_nearest = NearestNDInterpolator(pts, n_vals)
        self.replay_grid_interp = None

        try:
            df = pd.DataFrame({'V_ent': V_ent, 'V_exit': V_exit, 'n': n_vals})
            pivot = df.pivot_table(index='V_ent', columns='V_exit', values='n', aggfunc='mean')
            pivot = pivot.sort_index(axis=0).sort_index(axis=1)
            grid = pivot.to_numpy(dtype=float)
            if not np.isnan(grid).any() and grid.size > 0:
                ent_axis = pivot.index.to_numpy(dtype=float)
                exit_axis = pivot.columns.to_numpy(dtype=float)
                self.replay_grid_interp = RegularGridInterpolator(
                    (ent_axis, exit_axis),
                    grid,
                    bounds_error=False,
                    fill_value=np.nan,
                )
        except Exception:
            self.replay_grid_interp = None

        self.replay_vp_ref = float(getattr(self.cfg, 'SIM_REPLAY_V_P_REF', 0.0))
        self.replay_vp_sigma = max(float(getattr(self.cfg, 'SIM_REPLAY_V_P_SIGMA', 0.02)), 1e-6)
        self.noise_std = float(getattr(self.cfg, 'SIM_REPLAY_NOISE_STD_N', 0.0))

        self.replay_ent_bounds = (float(np.min(V_ent)), float(np.max(V_ent)))
        self.replay_exit_bounds = (float(np.min(V_exit)), float(np.max(V_exit)))
        self.replay_data_path = data_path
        self.simulation_model_kind = 'replay_2d'

        print(f'   Replay data loaded: {data_path.name} ({len(V_ent)} points)')
        print(f'   Replay bounds V_ENT:  [{self.replay_ent_bounds[0]:+.4f}, {self.replay_ent_bounds[1]:+.4f}] V')
        print(f'   Replay bounds V_EXIT: [{self.replay_exit_bounds[0]:+.4f}, {self.replay_exit_bounds[1]:+.4f}] V')
        print(f'   Replay V_P ref/sigma: {self.replay_vp_ref:+.4f} / {self.replay_vp_sigma:.4f} V')
        return True

    def _interp_replay_n(self, V_ent, V_exit):
        if self.replay_grid_interp is not None:
            v = self.replay_grid_interp(np.array([[V_ent, V_exit]], dtype=float))
            val = float(np.asarray(v).reshape(-1)[0])
            if np.isfinite(val):
                return val

        v_lin = self.replay_interp_linear(np.array([[V_ent, V_exit]], dtype=float))
        val_lin = float(np.asarray(v_lin).reshape(-1)[0])
        if np.isfinite(val_lin):
            return val_lin

        v_nn = self.replay_interp_nearest(np.array([[V_ent, V_exit]], dtype=float))
        return float(np.asarray(v_nn).reshape(-1)[0])

    def set_current_amp_gain(self, gain_a_per_v):
        self.current_amp_gain = float(gain_a_per_v)
        self.cfg.CURRENT_AMP_GAIN = float(gain_a_per_v)
        print(f'🔧 Active current amp gain updated: {self.current_amp_gain:.0e} A/V')

    def _write_voltages_direct(self, V_ent, V_p, V_exit):
        self.yoko_ent.write(f'S{V_ent:.6f}E')
        self.yoko_p.write(f'S{V_p:.6f}E')
        self.yoko_exit.write(f'S{V_exit:.6f}E')

    def set_voltages_stepwise(self, V_ent, V_p, V_exit):
        target = np.array([float(V_ent), float(V_p), float(V_exit)], dtype=float)
        start = np.array([
            self.current_voltages['V_ent'],
            self.current_voltages['V_p'],
            self.current_voltages['V_exit']
        ], dtype=float)

        max_delta = np.max(np.abs(target - start))
        n_steps = max(1, int(np.ceil(max_delta / self.cfg.MAX_VOLTAGE_STEP_V)))

        for k in range(1, n_steps + 1):
            alpha = k / n_steps
            vk = start + alpha * (target - start)

            if not self.simulation_mode:
                self._write_voltages_direct(vk[0], vk[1], vk[2])
                time.sleep(self.cfg.SETTLING_TIME)

            self.current_voltages['V_ent'] = float(vk[0])
            self.current_voltages['V_p'] = float(vk[1])
            self.current_voltages['V_exit'] = float(vk[2])

    def set_voltages(self, V_ent, V_p, V_exit):
        self.set_voltages_stepwise(V_ent, V_p, V_exit)

    def measure_current(self):
        if not self.simulation_mode:
            try:
                dmm_voltage = float(self.dmm.query('fetch?'))
                current = -dmm_voltage * self.current_amp_gain
                return current
            except Exception as e:
                print(f'⚠️ DMM error: {e}')
                return np.nan

        n = self._simulate_n(
            self.current_voltages['V_ent'],
            self.current_voltages['V_p'],
            self.current_voltages['V_exit']
        )
        return n * self.cfg.target_current

    def measure_current_at(self, V_ent, V_p, V_exit):
        self.set_voltages_stepwise(V_ent, V_p, V_exit)
        return self.measure_current()

    def measure(self, V_ent, V_p, V_exit):
        current = self.measure_current_at(V_ent, V_p, V_exit)
        return current / self.cfg.target_current

    def measure_2d(self, V_ent, V_exit, V_p_fixed):
        return self.measure(V_ent, V_p_fixed, V_exit)

    def _simulate_n(self, V_ent, V_p, V_exit):
        if getattr(self, 'simulation_model_kind', 'analytic') == 'replay_2d':
            n_2d = self._interp_replay_n(V_ent, V_exit)
            vp_factor = np.exp(-0.5 * ((V_p - self.replay_vp_ref) / self.replay_vp_sigma) ** 2)
            n = n_2d * vp_factor
            n += np.random.normal(0, self.noise_std)
            return n

        Va_eff = self.Va_base + self.cross_coupling_ent * (V_ent - self.V_ent_ref)
        arg1 = np.clip(-(V_exit - Va_eff) / self.Vb, -100, 100)
        arg2 = np.clip(-(V_exit - Va_eff + self.delta2) / self.Vb, -100, 100)
        f_exit = np.exp(-np.exp(arg1)) + np.exp(-np.exp(arg2))

        V_L = self.V_ent_center - self.ent_width / 2
        V_R = self.V_ent_center + self.ent_width / 2
        f_ent = self._sigmoid(self.ent_sharpness * (V_ent - V_L)) * self._sigmoid(-self.ent_sharpness * (V_ent - V_R))

        V_L_p = self.V_p_center - self.p_width / 2
        V_R_p = self.V_p_center + self.p_width / 2
        f_p = self._sigmoid(self.p_sharpness * (V_p - V_L_p)) * self._sigmoid(-self.p_sharpness * (V_p - V_R_p))

        n = f_exit * f_ent * f_p
        n += np.random.normal(0, self.noise_std)
        return n

    def _sigmoid(self, x):
        return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

    def get_ground_truth_map(self, V_ent_range, V_exit_range, V_p_fixed):
        if not self.simulation_mode:
            return None, None, None

        V_ENT, V_EXIT = np.meshgrid(V_ent_range, V_exit_range)
        n_map = np.zeros_like(V_ENT)

        noise_backup = self.noise_std
        self.noise_std = 0
        for i in range(len(V_exit_range)):
            for j in range(len(V_ent_range)):
                n_map[i, j] = self._simulate_n(V_ENT[i, j], V_p_fixed, V_EXIT[i, j])
        self.noise_std = noise_backup

        return V_ENT, V_EXIT, n_map

    def close(self):
        if not self.simulation_mode:
            try:
                self.yoko_ent.close()
                self.yoko_p.close()
                self.yoko_exit.close()
                self.dmm.close()
                self.rm.close()
                print('Instruments disconnected')
            except Exception:
                pass

    def get_mode_string(self):
        return 'SIMULATION' if self.simulation_mode else 'HARDWARE'


print('InstrumentController defined')

In [ ]:
# Cell 4: Bayesian Optimizer (for Phase 4)

class BayesianOptimizer:
    """BO with EI acquisition for Phase 1 optimization."""
    
    def __init__(self, bounds):
        self.bounds = bounds
        self.dim = len(bounds)
        
        self.kernel = ConstantKernel(1.0) * Matern(length_scale=0.1, nu=2.5) + \
                      WhiteKernel(noise_level=1e-5)
        
        self.gp_cost = GaussianProcessRegressor(
            kernel=self.kernel, n_restarts_optimizer=10, normalize_y=True)
        self.gp_n = GaussianProcessRegressor(
            kernel=self.kernel, n_restarts_optimizer=10, normalize_y=True)
        
        self.X_train = None
        self.y_train = None
        self.n_train = None
        self.is_fitted = False
    
    def fit(self, X, y_cost, n_values):
        self.X_train = np.array(X)
        self.y_train = np.array(y_cost)
        self.n_train = np.array(n_values)
        self.gp_cost.fit(self.X_train, self.y_train)
        self.gp_n.fit(self.X_train, self.n_train)
        self.is_fitted = True
    
    def _expected_improvement(self, X):
        X = np.atleast_2d(X)
        mu, sigma = self.gp_cost.predict(X, return_std=True)
        sigma = np.maximum(sigma, 1e-9)
        y_best = np.min(self.y_train)
        Z = (y_best - mu) / sigma
        ei = (y_best - mu) * norm.cdf(Z) + sigma * norm.pdf(Z)
        return -ei
    
    def suggest_next_point(self, n_candidates=5000):
        if not self.is_fitted:
            return np.random.uniform(self.bounds[:, 0], self.bounds[:, 1])
        
        X_candidates = np.random.uniform(
            self.bounds[:, 0], self.bounds[:, 1], size=(n_candidates, self.dim))
        ei_values = self._expected_improvement(X_candidates)
        best_idx = np.argmin(ei_values)
        x0 = X_candidates[best_idx]
        
        result = minimize(
            lambda x: self._expected_improvement(x.reshape(1, -1))[0],
            x0=x0, bounds=self.bounds, method='L-BFGS-B')
        return result.x
    
    def predict_cost(self, X):
        return self.gp_cost.predict(np.atleast_2d(X), return_std=True)
    
    def predict_n(self, X):
        return self.gp_n.predict(np.atleast_2d(X), return_std=True)


class EarlyStopping:
    def __init__(self, patience=25):
        self.patience = patience
        self.best_cost = np.inf
        self.counter = 0
    
    def update(self, cost):
        if cost < self.best_cost - 1e-6:
            self.best_cost = cost
            self.counter = 0
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience


print('BayesianOptimizer and EarlyStopping defined')

In [ ]:
# Cell 5: Efficient Mapper (for Phase 5)

class EfficientMapper:
    """GP-based mapper for Phase 5 with plateau-aware acquisition."""

    def __init__(self, bounds_2d, random_seed=42):
        self.bounds = np.asarray(bounds_2d, dtype=float)

        self.kernel = ConstantKernel(1.0) * Matern(length_scale=0.05, nu=2.5) + \
                      WhiteKernel(noise_level=1e-6)
        self.gp = GaussianProcessRegressor(
            kernel=self.kernel, n_restarts_optimizer=10, normalize_y=True)

        self.rng = np.random.default_rng(int(random_seed))
        self.X_measured = []
        self.y_measured = []
        self.sample_stage = []
        self.is_fitted = False

    def generate_lhs_samples(self, n_samples):
        """Generate Latin Hypercube samples."""
        seed = int(self.rng.integers(1, 1_000_000_000))
        sampler = qmc.LatinHypercube(d=2, seed=seed)
        samples = sampler.random(n=n_samples)
        lower = self.bounds[:, 0]
        upper = self.bounds[:, 1]
        return qmc.scale(samples, lower, upper)

    def fit(self):
        if len(self.X_measured) > 0:
            X = np.array(self.X_measured)
            y = np.array(self.y_measured)
            self.gp.fit(X, y)
            self.is_fitted = True

    def suggest_next_point(
        self,
        n_candidates=1000,
        target_levels=(1.0, 2.0),
        w_uncert=0.45,
        w_level=0.45,
        w_flat=0.10,
        tau_level=0.10,
        dv_flat=0.004,
        slope_tau=12.0,
    ):
        """
        Plateau-aware acquisition:
        score = w_u*uncertainty + w_l*level_proximity + w_f*flatness_proxy
        """
        if not self.is_fitted:
            x = self.rng.uniform(self.bounds[:, 0], self.bounds[:, 1], size=(2,))
            return x, {
                'score': np.nan,
                'sigma_norm': np.nan,
                'level_score': np.nan,
                'flat_score': np.nan,
            }

        candidates = self.rng.uniform(
            self.bounds[:, 0], self.bounds[:, 1], size=(int(n_candidates), 2)
        )

        mu, sigma = self.gp.predict(candidates, return_std=True)
        sigma = np.maximum(sigma, 1e-12)
        sigma_norm = sigma / (np.max(sigma) + 1e-12)

        tau_level = max(float(tau_level), 1e-6)
        target_levels = np.asarray(tuple(target_levels), dtype=float)
        if target_levels.size > 0:
            level_terms = [np.exp(-np.abs(mu - lv) / tau_level) for lv in target_levels]
            level_score = np.max(np.vstack(level_terms), axis=0)
        else:
            level_score = np.zeros_like(mu)

        dv = max(float(dv_flat), 1e-6)
        c_minus = candidates.copy()
        c_plus = candidates.copy()
        c_minus[:, 1] = np.clip(candidates[:, 1] - dv, self.bounds[1, 0], self.bounds[1, 1])
        c_plus[:, 1] = np.clip(candidates[:, 1] + dv, self.bounds[1, 0], self.bounds[1, 1])
        mu_minus = self.gp.predict(c_minus)
        mu_plus = self.gp.predict(c_plus)
        denom = np.maximum(c_plus[:, 1] - c_minus[:, 1], 1e-9)
        slope = np.abs((mu_plus - mu_minus) / denom)
        slope_tau = max(float(slope_tau), 1e-6)
        flat_score = np.exp(-slope / slope_tau)

        score = float(w_uncert) * sigma_norm + float(w_level) * level_score + float(w_flat) * flat_score
        idx = int(np.argmax(score))

        info = {
            'score': float(score[idx]),
            'sigma_norm': float(sigma_norm[idx]),
            'level_score': float(level_score[idx]),
            'flat_score': float(flat_score[idx]),
        }
        return candidates[idx], info

    def add_measurement(self, x, y, stage='sample'):
        self.X_measured.append(x)
        self.y_measured.append(y)
        self.sample_stage.append(stage)

    def predict_map(self, V_ent_range, V_exit_range):
        """Predict full 2D map."""
        V_ENT, V_EXIT = np.meshgrid(V_ent_range, V_exit_range)
        X_grid = np.column_stack([V_ENT.ravel(), V_EXIT.ravel()])
        n_mean, n_std = self.gp.predict(X_grid, return_std=True)
        return V_ENT, V_EXIT, n_mean.reshape(V_ENT.shape), n_std.reshape(V_ENT.shape)

    def predict_curve(self, V_exit_range, V_ent_fixed):
        """Predict pumping curve at fixed V_ENT."""
        X = np.column_stack([np.full_like(V_exit_range, V_ent_fixed), V_exit_range])
        return self.gp.predict(X, return_std=True)


print('EfficientMapper defined')

In [ ]:
# Cell 6: Phase 1 (Pinch-off) + Phase 4 (Constrained BO for n=1)


def _find_threshold_crossing(voltage_axis, abs_current_axis, threshold_a):
    """Find first crossing where |I| <= threshold. Fallback: argmin(|I|)."""
    below = abs_current_axis <= threshold_a
    if np.any(below):
        idx1 = int(np.argmax(below))
        if idx1 == 0:
            return float(voltage_axis[idx1]), idx1, idx1, True
        idx0 = idx1 - 1
        x0, x1 = voltage_axis[idx0], voltage_axis[idx1]
        y0, y1 = abs_current_axis[idx0], abs_current_axis[idx1]
        if np.isclose(y1, y0):
            x_cross = x1
        else:
            x_cross = x0 + (threshold_a - y0) * (x1 - x0) / (y1 - y0)
        return float(x_cross), idx0, idx1, True

    idx = int(np.argmin(abs_current_axis))
    return float(voltage_axis[idx]), idx, idx, False


def _expand_interval(interval, margin):
    lo, hi = float(interval[0]), float(interval[1])
    return np.array([lo - float(margin), hi + float(margin)], dtype=float)


def _clip_interval(interval, clip_range):
    lo = max(float(interval[0]), float(clip_range[0]))
    hi = min(float(interval[1]), float(clip_range[1]))
    if lo > hi:
        mid = 0.5 * (lo + hi)
        lo = hi = mid
    return np.array([lo, hi], dtype=float)


def _intersect_interval(a, b):
    lo = max(float(a[0]), float(b[0]))
    hi = min(float(a[1]), float(b[1]))
    if lo <= hi:
        return np.array([lo, hi], dtype=float), True
    return np.array([lo, lo], dtype=float), False


def run_phase1_pinchoff(instr, config):
    """
    Phase 1: find pinch-off voltages of V_ent, V_p, V_exit at gain=1e-7 A/V.
    """
    print()
    print('='*70)
    print('PHASE 1: PINCH-OFF SEARCH')
    print('Gain set to 1e-7 A/V, sweep each gate independently')
    print('='*70)

    instr.set_current_amp_gain(config.GAIN_PHASE1_A_PER_V)

    refs = {
        'V_ent': float(config.PINCH_REF_V_ENT),
        'V_p': float(config.PINCH_REF_V_P),
        'V_exit': float(config.PINCH_REF_V_EXIT),
    }

    gate_keys = ['V_ent', 'V_p', 'V_exit']
    traces = {}
    pinch_off = {}
    total_measurements = 0

    for gate in gate_keys:
        vmin, vmax = config.PINCH_SCAN_RANGES[gate]
        sweep = np.linspace(vmin, vmax, config.PINCH_SCAN_POINTS)
        currents = []

        print()
        print(f'[Phase1] Sweeping {gate}: {vmin:.3f} -> {vmax:.3f} V ({config.PINCH_SCAN_POINTS} points)')

        for i, vg in enumerate(sweep):
            V_ent = refs['V_ent']
            V_p = refs['V_p']
            V_exit = refs['V_exit']

            if gate == 'V_ent':
                V_ent = vg
            elif gate == 'V_p':
                V_p = vg
            else:
                V_exit = vg

            current = instr.measure_current_at(V_ent, V_p, V_exit)
            currents.append(current)
            total_measurements += 1

            if (i + 1) % 30 == 0 or i == len(sweep) - 1:
                print(f'  {gate} {i+1:3d}/{len(sweep)}: V={vg:+.4f} V, I={current*1e9:+.4f} nA')

        currents = np.asarray(currents, dtype=float)
        abs_currents = np.abs(currents)

        pinch_v, idx0, idx1, crossed = _find_threshold_crossing(
            sweep, abs_currents, config.PINCH_OFF_THRESHOLD_A
        )

        pinch_off[gate] = float(pinch_v)
        traces[gate] = {
            'voltage': sweep,
            'current_a': currents,
            'abs_current_a': abs_currents,
            'cross_idx0': int(idx0),
            'cross_idx1': int(idx1),
            'crossed_threshold': bool(crossed),
            'pinch_off_v': float(pinch_v),
        }

        flag = 'threshold-cross' if crossed else 'fallback-min|I|'
        print(f'  -> Pinch-off {gate}: {pinch_v:+.6f} V ({flag})')

    results = {
        'phase': 'phase1_pinchoff',
        'gain_a_per_v': float(instr.current_amp_gain),
        'pinch_threshold_a': float(config.PINCH_OFF_THRESHOLD_A),
        'pinch_off_voltages': pinch_off,
        'traces': traces,
        'total_measurements': int(total_measurements),
        'refs': refs,
    }
    return results


def display_phase1_results(results, config):
    print()
    print('='*70)
    print('PHASE 1 COMPLETE - PINCH-OFF RESULTS')
    print('='*70)
    print(f'Gain used:        {results["gain_a_per_v"]:.0e} A/V')
    print(f'|I| threshold:    {results["pinch_threshold_a"]*1e9:.3f} nA')
    print(f'Measurements:     {results["total_measurements"]}')

    po = results['pinch_off_voltages']
    print()
    print('Pinch-off voltages:')
    print(f'  V_ENT pinch-off:  {po["V_ent"]:+.6f} V')
    print(f'  V_P pinch-off:    {po["V_p"]:+.6f} V')
    print(f'  V_EXIT pinch-off: {po["V_exit"]:+.6f} V')

    V_ent_center = po['V_ent'] + config.PHASE2_CENTER_OFFSET_V_ENT
    V_exit_center = po['V_exit'] + config.PHASE2_CENTER_OFFSET_V_EXIT

    if getattr(config, 'USE_FIXED_V_P_MODE', False):
        V_p_center = float(config.FIXED_V_P_VALUE) + config.PHASE2_CENTER_OFFSET_V_P
        half_p = max(float(config.FIXED_V_P_PHASE2_RANGE), 0.0) / 2
    else:
        V_p_center = po['V_p'] + config.PHASE2_CENTER_OFFSET_V_P
        half_p = config.PHASE2_RANGE_V_P / 2

    half_ent = config.PHASE2_RANGE_V_ENT / 2
    half_exit = config.PHASE2_RANGE_V_EXIT / 2

    print()
    print('Proposed Phase 2 dynamic search cube (sparse 3D):')
    print(f'  Center V_ENT: {V_ent_center:+.4f} V')
    print(f'  Center V_P:   {V_p_center:+.4f} V')
    print(f'  Center V_EXIT:{V_exit_center:+.4f} V')
    print(f'  V_ENT:  [{V_ent_center-half_ent:+.3f}, {V_ent_center+half_ent:+.3f}] V')
    print(f'  V_P:    [{V_p_center-half_p:+.3f}, {V_p_center+half_p:+.3f}] V')
    print(f'  V_EXIT: [{V_exit_center-half_exit:+.3f}, {V_exit_center+half_exit:+.3f}] V')


def run_phase4_optimization(instr, phase1_results, phase2_selected, config):
    """
    Phase 4: constrained BO to find n=1 region.
    Bounds are dynamically derived from Phase 2 valid region.
    """
    print()
    print('='*70)
    print('PHASE 4: CONSTRAINED BO FOR n=1 (DYNAMIC BOUNDS)')
    print('='*70)

    if config.PHASE4_USE_PHASE2_VALID_BOUNDS:
        base_bounds_3d = np.asarray(phase2_selected['valid_bounds_3d'], dtype=float)
        bounds_tag = 'phase2_valid_bounds_3d'
    else:
        base_bounds_3d = np.asarray(phase2_selected['bounds_3d_default'], dtype=float)
        bounds_tag = 'phase2_default_bounds_3d'

    default_bounds_3d = np.asarray(phase2_selected['bounds_3d_default'], dtype=float)

    V_ent_bounds = _expand_interval(base_bounds_3d[0], config.PHASE4_MARGIN_V_ENT)
    V_p_bounds_base = _expand_interval(base_bounds_3d[1], config.PHASE4_MARGIN_V_P)
    V_exit_bounds = _expand_interval(base_bounds_3d[2], config.PHASE4_MARGIN_V_EXIT)

    V_ent_bounds = _clip_interval(V_ent_bounds, default_bounds_3d[0])
    V_p_bounds_base = _clip_interval(V_p_bounds_base, default_bounds_3d[1])
    V_exit_bounds = _clip_interval(V_exit_bounds, default_bounds_3d[2])

    if getattr(config, 'USE_FIXED_V_P_MODE', False):
        V_p_fixed = float(config.FIXED_V_P_VALUE)
        V_p_bounds = _clip_interval(np.array([V_p_fixed, V_p_fixed], dtype=float), default_bounds_3d[1])
        bounds_tag = bounds_tag + '+fixed_vp'
    else:
        if 'best_point' in phase2_selected and phase2_selected['best_point'] is not None:
            V_p_center_ref = float(phase2_selected['best_point'][1])
        else:
            V_p_center_ref = float(phase2_selected['V_p_center'])

        V_p_center = V_p_center_ref + float(config.PHASE4_VP_OFFSET)
        half_vp = config.PHASE4_RANGE_V_P / 2
        V_p_window = np.array([V_p_center - half_vp, V_p_center + half_vp], dtype=float)
        V_p_window = _clip_interval(V_p_window, default_bounds_3d[1])

        V_p_bounds, ok_intersection = _intersect_interval(V_p_bounds_base, V_p_window)
        if not ok_intersection:
            V_p_bounds = V_p_window

    bounds_optimization = np.array([
        [V_ent_bounds[0], V_ent_bounds[1]],
        [V_p_bounds[0], V_p_bounds[1]],
        [V_exit_bounds[0], V_exit_bounds[1]],
    ], dtype=float)

    print(f'Bounds source: {bounds_tag}')
    print('Constrained BO bounds:')
    print(f'  V_ENT:  [{bounds_optimization[0,0]:+.4f}, {bounds_optimization[0,1]:+.4f}] V')
    print(f'  V_P:    [{bounds_optimization[1,0]:+.4f}, {bounds_optimization[1,1]:+.4f}] V')
    print(f'  V_EXIT: [{bounds_optimization[2,0]:+.4f}, {bounds_optimization[2,1]:+.4f}] V')

    bo = BayesianOptimizer(bounds_optimization)
    early_stop = EarlyStopping(patience=config.PHASE4_EARLY_STOP_PATIENCE)

    X_hist, y_hist, n_hist = [], [], []

    print()
    print(f'{"Iter":>5} {"Phase":>6} {"V_ENT":>9} {"V_P":>9} {"V_EXIT":>9} {"n":>10} {"Cost":>8} {"Best":>8}')
    print('-'*75)

    total_iterations = config.PHASE4_N_INITIAL + config.PHASE4_N_ITER
    stopped_early = False
    early_stop_iteration = None

    for i in range(total_iterations):
        if i < config.PHASE4_N_INITIAL:
            phase = 'Init'
            x_next = np.random.uniform(bounds_optimization[:, 0], bounds_optimization[:, 1])
        else:
            phase = 'BO'
            bo.fit(X_hist, y_hist, n_hist)
            x_next = bo.suggest_next_point()

        V_ent, V_p, V_exit = x_next
        n = instr.measure(V_ent, V_p, V_exit)
        cost = np.log10(np.abs(n - config.PHASE4_TARGET_N) + 1e-12)

        X_hist.append(x_next)
        y_hist.append(cost)
        n_hist.append(n)

        best_cost = np.min(y_hist)
        print(f'{i:>5} {phase:>6} {V_ent:>9.4f} {V_p:>9.4f} {V_exit:>9.4f} {n:>10.5f} {cost:>8.2f} {best_cost:>8.2f}')

        if phase == 'BO':
            if early_stop.update(cost):
                early_stop_iteration = i
                stopped_early = True
                print()
                print(f'⚠️ Early stopping at iteration {i}')
                break

    bo.fit(X_hist, y_hist, n_hist)

    X_hist_arr = np.array(X_hist)
    y_hist_arr = np.array(y_hist)
    n_hist_arr = np.array(n_hist)

    best_idx_by_cost = int(np.argmin(y_hist_arr))
    best_idx = best_idx_by_cost

    plateau_scan = []
    phase4_refinement_measurements = 0
    phase4_refinement_used = False

    top_k = min(config.PHASE4_PLATEAU_TOP_K, len(X_hist_arr))
    if top_k > 0:
        n_error = np.abs(n_hist_arr - config.PHASE4_TARGET_N)
        candidate_idx = np.argsort(n_error)[:top_k]

        near_plateau_idx = candidate_idx[n_error[candidate_idx] <= config.PHASE4_N_TOL]
        if len(near_plateau_idx) > 0:
            candidate_idx = near_plateau_idx

        v_exit_min, v_exit_max = bounds_optimization[2]

        for idx in candidate_idx:
            V_ent, V_p, V_exit = X_hist_arr[idx]
            n_center = n_hist_arr[idx]

            v_minus = np.clip(V_exit - config.PHASE4_DV_EXIT, v_exit_min, v_exit_max)
            v_plus = np.clip(V_exit + config.PHASE4_DV_EXIT, v_exit_min, v_exit_max)
            if np.isclose(v_plus, v_minus):
                continue

            n_minus = instr.measure(V_ent, V_p, v_minus)
            n_plus = instr.measure(V_ent, V_p, v_plus)
            phase4_refinement_measurements += 2

            dn_dV_exit = (n_plus - n_minus) / (v_plus - v_minus)
            flatness = np.abs(n_plus - n_minus)
            n_error_center = np.abs(n_center - config.PHASE4_TARGET_N)
            plateau_score = n_error_center + config.PHASE4_FLATNESS_WEIGHT * flatness

            plateau_scan.append({
                'idx': int(idx),
                'V_ent': float(V_ent),
                'V_p': float(V_p),
                'V_exit': float(V_exit),
                'n_center': float(n_center),
                'n_minus': float(n_minus),
                'n_plus': float(n_plus),
                'dn_dV_exit': float(dn_dV_exit),
                'flatness': float(flatness),
                'n_error': float(n_error_center),
                'plateau_score': float(plateau_score),
            })

    if plateau_scan:
        best_plateau_row = min(plateau_scan, key=lambda row: row['plateau_score'])
        best_idx = int(best_plateau_row['idx'])
        phase4_refinement_used = True

    best_x = X_hist_arr[best_idx]
    best_n = n_hist_arr[best_idx]
    best_cost = y_hist_arr[best_idx]

    best_dn_dV_exit = np.nan
    best_plateau_score = np.nan
    for row in plateau_scan:
        if row['idx'] == best_idx:
            best_dn_dV_exit = row['dn_dV_exit']
            best_plateau_score = row['plateau_score']
            break

    results = {
        'phase': 'phase4_n1_optimization',
        'bounds_source': bounds_tag,
        'bounds_optimization': bounds_optimization,
        'X_hist': X_hist_arr,
        'y_hist': y_hist_arr,
        'n_hist': n_hist_arr,
        'best_V_ent': float(best_x[0]),
        'best_V_p': float(best_x[1]),
        'best_V_exit': float(best_x[2]),
        'best_n': float(best_n),
        'best_cost': float(best_cost),
        'best_dn_dV_exit': float(best_dn_dV_exit),
        'best_plateau_score': float(best_plateau_score),
        'best_idx': int(best_idx),
        'best_idx_by_cost': int(best_idx_by_cost),
        'phase4_target_n': float(config.PHASE4_TARGET_N),
        'phase1_target_n': float(config.PHASE4_TARGET_N),
        'phase4_refinement_used': phase4_refinement_used,
        'phase4_refinement_candidates': len(plateau_scan),
        'phase4_refinement_measurements': int(phase4_refinement_measurements),
        'total_measurements_including_refinement': int(len(X_hist_arr) + phase4_refinement_measurements),
        'plateau_scan': plateau_scan,
        'bo': bo,
        'stopped_early': bool(stopped_early),
        'early_stop_iteration': early_stop_iteration,
        'total_iterations': len(X_hist_arr),
        'selected_phase2_attempt': int(phase2_selected['attempt_idx']),
    }
    return results


def display_phase4_results(results, config):
    print()
    print('='*70)
    print('PHASE 4 COMPLETE - n=1 OPTIMIZATION RESULTS')
    print('='*70)

    print(f'Bounds source:       {results.get("bounds_source", "N/A")}')
    print(f'Total measurements:  {results["total_measurements_including_refinement"]}')
    print(f'Best V_ENT:          {results["best_V_ent"]:+.6f} V')
    print(f'Best V_P:            {results["best_V_p"]:+.6f} V')
    print(f'Best V_EXIT:         {results["best_V_exit"]:+.6f} V')
    print(f'Best n:              {results["best_n"]:.6f}')
    print(f'Best |n-1|:          {np.abs(results["best_n"] - results["phase4_target_n"]):.3e}')

    if results['phase4_refinement_used']:
        print(f'Refinement used:     Yes ({results["phase4_refinement_candidates"]} candidates)')
    else:
        print('Refinement used:     No')

In [ ]:
# Cell 7: Phase 2 (Sparse Adaptive 3D) + Phase 5 (Pump Map)


def _build_phase2_3d_bounds(phase1_results, config):
    po = phase1_results['pinch_off_voltages']

    if config.PHASE2_CENTER_FROM_PHASE1_PINCH:
        V_ent_center = float(po['V_ent']) + float(config.PHASE2_CENTER_OFFSET_V_ENT)
        V_p_center = float(po['V_p']) + float(config.PHASE2_CENTER_OFFSET_V_P)
        V_exit_center = float(po['V_exit']) + float(config.PHASE2_CENTER_OFFSET_V_EXIT)
        center_source = 'phase1_pinch_off'
    else:
        V_ent_center = float(config.PINCH_REF_V_ENT)
        V_p_center = float(config.PINCH_REF_V_P)
        V_exit_center = float(config.PINCH_REF_V_EXIT)
        center_source = 'config_ref'

    if getattr(config, 'USE_FIXED_V_P_MODE', False):
        V_p_center = float(config.FIXED_V_P_VALUE) + float(config.PHASE2_CENTER_OFFSET_V_P)
        half_p = max(float(config.FIXED_V_P_PHASE2_RANGE), 0.0) / 2
        center_source = center_source + '+fixed_vp'
    else:
        half_p = config.PHASE2_RANGE_V_P / 2

    half_ent = config.PHASE2_RANGE_V_ENT / 2
    half_exit = config.PHASE2_RANGE_V_EXIT / 2

    bounds_3d_default = np.array([
        [V_ent_center - half_ent, V_ent_center + half_ent],
        [V_p_center - half_p, V_p_center + half_p],
        [V_exit_center - half_exit, V_exit_center + half_exit],
    ], dtype=float)

    return bounds_3d_default, np.array([V_ent_center, V_p_center, V_exit_center], dtype=float), center_source


def _phase2_fit_gp(X, y, length_scale=0.05, noise=1e-8):
    kernel = ConstantKernel(1.0) * Matern(length_scale=length_scale, nu=2.5) + WhiteKernel(noise_level=noise)
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=3, normalize_y=True)
    gp.fit(X, y)
    return gp


def run_phase2_positive_current_mapping(instr, phase1_results, config, attempt_idx=1):
    """
    Phase 2: sparse adaptive 3D search for region satisfying 0 < I < 4 nA.
    Variables: V_ENT, V_P, V_EXIT are treated equally as search dimensions.
    """
    print()
    print('='*70)
    print(f'PHASE 2: SPARSE ADAPTIVE 3D SEARCH (Attempt {attempt_idx})')
    print('='*70)

    instr.set_current_amp_gain(config.GAIN_PHASE2_A_PER_V)

    bounds_3d_default, center_3d, center_source = _build_phase2_3d_bounds(phase1_results, config)

    print(f'Center source: {center_source}')
    print(f'  Center V_ENT: {center_3d[0]:+.4f} V')
    print(f'  Center V_P:   {center_3d[1]:+.4f} V')
    print(f'  Center V_EXIT:{center_3d[2]:+.4f} V')
    print(f'  V_ENT range:  [{bounds_3d_default[0,0]:+.4f}, {bounds_3d_default[0,1]:+.4f}] V')
    print(f'  V_P range:    [{bounds_3d_default[1,0]:+.4f}, {bounds_3d_default[1,1]:+.4f}] V')
    print(f'  V_EXIT range: [{bounds_3d_default[2,0]:+.4f}, {bounds_3d_default[2,1]:+.4f}] V')
    if getattr(config, 'USE_FIXED_V_P_MODE', False):
        print(f'  Fixed V_P preset ON: V_P={config.FIXED_V_P_VALUE:+.4f} V (Phase2 V_P range={config.FIXED_V_P_PHASE2_RANGE:.4f} V)')

    rng = np.random.default_rng(config.PHASE2_RANDOM_SEED + int(attempt_idx))
    sampler = qmc.LatinHypercube(d=3, seed=config.PHASE2_RANDOM_SEED + int(attempt_idx))
    lhs_samples = sampler.random(n=config.PHASE2_N_INITIAL_3D)

    lower = bounds_3d_default[:, 0].copy()
    upper = bounds_3d_default[:, 1].copy()
    flat_dims = upper <= lower
    upper_qmc = upper.copy()
    upper_qmc[flat_dims] = lower[flat_dims] + 1e-12

    init_points = qmc.scale(lhs_samples, lower, upper_qmc)
    if np.any(flat_dims):
        init_points[:, flat_dims] = lower[flat_dims]

    X_measured = []
    I_measured_a = []
    n_measured = []
    sample_stage = []
    adaptive_score = []

    print()
    print(f'Phase 2a: Initial 3D LHS sampling ({config.PHASE2_N_INITIAL_3D} points)')
    print('-'*50)

    for i, x in enumerate(init_points):
        V_ent, V_p, V_exit = x
        I = instr.measure_current_at(V_ent, V_p, V_exit)
        n = I / config.target_current

        X_measured.append([V_ent, V_p, V_exit])
        I_measured_a.append(I)
        n_measured.append(n)
        sample_stage.append('init')
        adaptive_score.append(np.nan)

        if (i + 1) % 10 == 0 or (i + 1) == config.PHASE2_N_INITIAL_3D:
            print(f'  Init {i+1:3d}/{config.PHASE2_N_INITIAL_3D}: V_ENT={V_ent:+.3f}V, V_P={V_p:+.3f}V, V_EXIT={V_exit:+.3f}V, I={I*1e9:+.4f}nA, n={n:+.4f}')

    print()
    print(f'Phase 2b: Adaptive sparse 3D sampling ({config.PHASE2_N_ADAPTIVE_3D} points)')
    print('-'*50)

    tau_n = max(float(config.PHASE2_SCORE_TAU_N), 1e-6)
    for i in range(config.PHASE2_N_ADAPTIVE_3D):
        X_arr = np.asarray(X_measured, dtype=float)
        I_arr = np.asarray(I_measured_a, dtype=float)
        n_arr = np.asarray(n_measured, dtype=float)

        gp_I = _phase2_fit_gp(X_arr, I_arr, length_scale=0.06, noise=1e-9)
        gp_n = _phase2_fit_gp(X_arr, n_arr, length_scale=0.06, noise=1e-6)

        cand = rng.uniform(lower, upper_qmc, size=(config.PHASE2_N_CANDIDATES_3D, 3))
        if np.any(flat_dims):
            cand[:, flat_dims] = lower[flat_dims]

        mu_I, sig_I = gp_I.predict(cand, return_std=True)
        mu_n, sig_n = gp_n.predict(cand, return_std=True)

        sig_I = np.maximum(sig_I, 1e-12)
        sig_n = np.maximum(sig_n, 1e-12)

        z_hi = (config.PHASE2_I_MAX_A - mu_I) / sig_I
        z_lo = (config.PHASE2_I_MIN_A - mu_I) / sig_I
        p_feasible = norm.cdf(z_hi) - norm.cdf(z_lo)

        n_quality = np.exp(-np.abs(mu_n - config.PHASE4_TARGET_N) / tau_n)
        sigma_n_norm = sig_n / (np.max(sig_n) + 1e-12)

        score = p_feasible * n_quality + float(config.PHASE2_SCORE_UNCERT_WEIGHT) * sigma_n_norm
        idx_best = int(np.argmax(score))
        x_next = cand[idx_best]

        V_ent, V_p, V_exit = x_next
        I = instr.measure_current_at(V_ent, V_p, V_exit)
        n = I / config.target_current

        X_measured.append([V_ent, V_p, V_exit])
        I_measured_a.append(I)
        n_measured.append(n)
        sample_stage.append('adaptive')
        adaptive_score.append(float(score[idx_best]))

        if (i + 1) % 10 == 0 or (i + 1) == config.PHASE2_N_ADAPTIVE_3D:
            print(f'  Adaptive {i+1:3d}/{config.PHASE2_N_ADAPTIVE_3D}: V_ENT={V_ent:+.3f}V, V_P={V_p:+.3f}V, V_EXIT={V_exit:+.3f}V, I={I*1e9:+.4f}nA, n={n:+.4f}, score={score[idx_best]:.4f}')

    X_measured = np.asarray(X_measured, dtype=float)
    I_measured_a = np.asarray(I_measured_a, dtype=float)
    n_measured = np.asarray(n_measured, dtype=float)
    adaptive_score = np.asarray(adaptive_score, dtype=float)

    valid_mask_points = (I_measured_a > config.PHASE2_I_MIN_A) & (I_measured_a < config.PHASE2_I_MAX_A)
    n_valid = int(np.sum(valid_mask_points))

    if n_valid > 0:
        X_valid = X_measured[valid_mask_points]
        valid_bounds_3d = np.array([
            [np.min(X_valid[:, 0]), np.max(X_valid[:, 0])],
            [np.min(X_valid[:, 1]), np.max(X_valid[:, 1])],
            [np.min(X_valid[:, 2]), np.max(X_valid[:, 2])],
        ], dtype=float)
        used_fallback_bounds = False

        idx_valid = np.where(valid_mask_points)[0]
        idx_best_valid = idx_valid[int(np.argmin(np.abs(n_measured[idx_valid] - config.PHASE4_TARGET_N)))]
    else:
        valid_bounds_3d = bounds_3d_default.copy()
        used_fallback_bounds = True
        idx_best_valid = int(np.argmin(np.abs(n_measured - config.PHASE4_TARGET_N)))

    best_point = X_measured[idx_best_valid]

    results = {
        'phase': 'phase2_positive_region_sparse3d',
        'attempt_idx': int(attempt_idx),
        'gain_a_per_v': float(instr.current_amp_gain),
        'center_source': center_source,
        'center_3d': center_3d,
        'V_ent_center': float(center_3d[0]),
        'V_p_center': float(center_3d[1]),
        'V_exit_center': float(center_3d[2]),
        'V_p_fixed': float(center_3d[1]),
        'fixed_v_p_mode': bool(getattr(config, 'USE_FIXED_V_P_MODE', False)),
        'fixed_v_p_value': float(getattr(config, 'FIXED_V_P_VALUE', np.nan)) if getattr(config, 'USE_FIXED_V_P_MODE', False) else np.nan,
        'bounds_3d_default': bounds_3d_default,
        'bounds_2d_default': np.array([[bounds_3d_default[0,0], bounds_3d_default[0,1]],
                                       [bounds_3d_default[2,0], bounds_3d_default[2,1]]], dtype=float),
        'valid_bounds_3d': valid_bounds_3d,
        'valid_bounds': np.array([[valid_bounds_3d[0,0], valid_bounds_3d[0,1]],
                                  [valid_bounds_3d[2,0], valid_bounds_3d[2,1]]], dtype=float),
        'X_measured': X_measured,
        'I_measured_a': I_measured_a,
        'I_measured_nA': I_measured_a * 1e9,
        'n_measured': n_measured,
        'sample_stage': np.asarray(sample_stage),
        'adaptive_score': adaptive_score,
        'valid_mask_points': valid_mask_points,
        'valid_points': n_valid,
        'valid_points_connected': n_valid,
        'used_fallback_bounds': bool(used_fallback_bounds),
        'i_condition_min_a': float(config.PHASE2_I_MIN_A),
        'i_condition_max_a': float(config.PHASE2_I_MAX_A),
        'best_point': best_point,
        'best_point_n': float(n_measured[idx_best_valid]),
        'best_point_I_A': float(I_measured_a[idx_best_valid]),
    }

    print()
    print('Phase 2 result summary (sparse 3D):')
    print(f'  Gain:                    {results["gain_a_per_v"]:.0e} A/V')
    print(f'  Total measured points:   {len(results["X_measured"])}')
    print(f'  Valid points:            {results["valid_points"]}')
    print(f'  Using fallback bounds:   {results["used_fallback_bounds"]}')
    print(f'  Valid V_ENT bounds:      [{results["valid_bounds_3d"][0,0]:+.4f}, {results["valid_bounds_3d"][0,1]:+.4f}] V')
    print(f'  Valid V_P bounds:        [{results["valid_bounds_3d"][1,0]:+.4f}, {results["valid_bounds_3d"][1,1]:+.4f}] V')
    print(f'  Valid V_EXIT bounds:     [{results["valid_bounds_3d"][2,0]:+.4f}, {results["valid_bounds_3d"][2,1]:+.4f}] V')
    print(f'  Best candidate n:        {results["best_point_n"]:+.5f}')

    return results


def run_phase5_mapping(instr, phase4_results, config, phase2_selected=None):
    """
    Phase 5: pump map mapping with
    1) bounds clipping (phase2 valid + replay bounds),
    2) plateau-aware adaptive sampling,
    3) optional dense line refinement.
    """
    print()
    print('='*70)
    print('PHASE 5: PUMP MAP MAPPING')
    print('Mapping pump characteristics around Phase 4 optimum')
    print('='*70)

    half_V_ent = config.MAPPING_RANGE_V_ENT / 2
    half_V_exit = config.MAPPING_RANGE_V_EXIT / 2

    V_ent_center = phase4_results['best_V_ent']
    if getattr(config, 'USE_FIXED_V_P_MODE', False):
        V_p_fixed = float(config.FIXED_V_P_VALUE)
    else:
        V_p_fixed = phase4_results['best_V_p']
    V_exit_center = phase4_results['best_V_exit']

    bounds_2d_base = np.array([
        [V_ent_center - half_V_ent, V_ent_center + half_V_ent],
        [V_exit_center - half_V_exit, V_exit_center + half_V_exit]
    ], dtype=float)
    bounds_2d = bounds_2d_base.copy()

    clip_notes = ['base(center ± configured range)']

    # Clip to phase2 valid bounds when available
    phase2_bounds_used = None
    if getattr(config, 'PHASE5_CLIP_TO_PHASE2_VALID_BOUNDS', True) and (phase2_selected is not None):
        if 'valid_bounds' in phase2_selected:
            phase2_bounds = np.asarray(phase2_selected['valid_bounds'], dtype=float)
            phase2_bounds_used = phase2_bounds.copy()
            for d in range(2):
                lo = max(bounds_2d[d, 0], phase2_bounds[d, 0])
                hi = min(bounds_2d[d, 1], phase2_bounds[d, 1])
                if lo <= hi:
                    bounds_2d[d] = [lo, hi]
            clip_notes.append('clipped_to_phase2_valid_bounds')

    # Clip to replay data bounds in replay_2d simulation
    replay_bounds_used = None
    if getattr(config, 'PHASE5_CLIP_TO_REPLAY_BOUNDS', True):
        if getattr(instr, 'simulation_mode', False) and getattr(instr, 'simulation_model_kind', '') == 'replay_2d':
            if hasattr(instr, 'replay_ent_bounds') and hasattr(instr, 'replay_exit_bounds'):
                replay_bounds = np.array([
                    [float(instr.replay_ent_bounds[0]), float(instr.replay_ent_bounds[1])],
                    [float(instr.replay_exit_bounds[0]), float(instr.replay_exit_bounds[1])],
                ], dtype=float)
                replay_bounds_used = replay_bounds.copy()
                for d in range(2):
                    lo = max(bounds_2d[d, 0], replay_bounds[d, 0])
                    hi = min(bounds_2d[d, 1], replay_bounds[d, 1])
                    if lo <= hi:
                        bounds_2d[d] = [lo, hi]
                clip_notes.append('clipped_to_replay_bounds')

    # Safety fallback if interval collapses
    for d in range(2):
        if not np.isfinite(bounds_2d[d, 0]) or not np.isfinite(bounds_2d[d, 1]) or bounds_2d[d, 1] <= bounds_2d[d, 0]:
            bounds_2d[d] = bounds_2d_base[d]
            clip_notes.append(f'fallback_base_dim{d}')

    print()
    print('Mapping parameters:')
    print(f'  Center V_ENT:  {V_ent_center:.4f} V')
    print(f'  Center V_EXIT: {V_exit_center:.4f} V')
    print(f'  Fixed V_P:     {V_p_fixed:.4f} V')
    print(f'  Base V_ENT range:  [{bounds_2d_base[0,0]:.3f}, {bounds_2d_base[0,1]:.3f}] V')
    print(f'  Base V_EXIT range: [{bounds_2d_base[1,0]:.3f}, {bounds_2d_base[1,1]:.3f}] V')
    print(f'  Final V_ENT range: [{bounds_2d[0,0]:.3f}, {bounds_2d[0,1]:.3f}] V')
    print(f'  Final V_EXIT range:[{bounds_2d[1,0]:.3f}, {bounds_2d[1,1]:.3f}] V')
    print(f'  Bounds notes:      {", ".join(clip_notes)}')

    mapper = EfficientMapper(bounds_2d, random_seed=config.PHASE2_RANDOM_SEED)

    print()
    print(f'Phase 5a: Latin Hypercube Sampling ({config.N_MAPPING_LHS} points)')
    print('-'*50)
    lhs_points = mapper.generate_lhs_samples(config.N_MAPPING_LHS)

    for i, (V_ent, V_exit) in enumerate(lhs_points):
        n = instr.measure_2d(V_ent, V_exit, V_p_fixed)
        mapper.add_measurement([V_ent, V_exit], n, stage='lhs')

        if (i + 1) % 10 == 0:
            print(f'  LHS {i+1:3d}/{config.N_MAPPING_LHS}: V_ENT={V_ent:.3f}V, V_EXIT={V_exit:.3f}V, n={n:.4f}')

    mapper.fit()
    print(f'  GP fitted with {len(mapper.X_measured)} points')

    print()
    print(f'Phase 5b: Plateau-aware adaptive sampling ({config.N_MAPPING_ADAPTIVE} points)')
    print('-'*50)

    for i in range(config.N_MAPPING_ADAPTIVE):
        next_point, acq_info = mapper.suggest_next_point(
            n_candidates=config.PHASE5_SUGGESTION_CANDIDATES,
            target_levels=config.PHASE5_ACQ_LEVELS,
            w_uncert=config.PHASE5_ACQ_W_UNCERT,
            w_level=config.PHASE5_ACQ_W_LEVEL,
            w_flat=config.PHASE5_ACQ_W_FLAT,
            tau_level=config.PHASE5_ACQ_LEVEL_TAU,
            dv_flat=config.PHASE5_ACQ_FLAT_DV,
            slope_tau=config.PHASE5_ACQ_FLAT_SLOPE_TAU,
        )
        V_ent, V_exit = next_point

        n = instr.measure_2d(V_ent, V_exit, V_p_fixed)
        mapper.add_measurement([V_ent, V_exit], n, stage='adaptive')
        mapper.fit()

        if (i + 1) % 10 == 0:
            V_ent_test = np.linspace(bounds_2d[0, 0], bounds_2d[0, 1], 20)
            V_exit_test = np.linspace(bounds_2d[1, 0], bounds_2d[1, 1], 20)
            _, _, _, sigma_map = mapper.predict_map(V_ent_test, V_exit_test)
            mean_sigma = np.mean(sigma_map)

            print(
                f'  Adaptive {i+1:3d}/{config.N_MAPPING_ADAPTIVE}: '
                f'VENT={V_ent:.3f}V, VEXIT={V_exit:.3f}V, n={n:.4f}, '
                f'mean_σ={mean_sigma:.4f}, acq={acq_info["score"]:.3f}'
            )

    refinement_lines = []
    refinement_points = 0

    if getattr(config, 'PHASE5_ENABLE_DENSE_REFINEMENT', True):
        print()
        print('Phase 5c: Dense line refinement')
        print('-'*50)

        X_arr = np.array(mapper.X_measured)
        y_arr = np.array(mapper.y_measured)

        candidate_lines = []
        for lvl in tuple(getattr(config, 'PHASE5_REFINEMENT_TARGET_LEVELS', (1.0, 2.0))):
            idx = int(np.argmin(np.abs(y_arr - float(lvl))))
            candidate_lines.append((float(X_arr[idx, 0]), float(X_arr[idx, 1]), float(lvl), 'target'))

        candidate_lines.append((float(V_ent_center), float(V_exit_center), 1.0, 'center'))

        selected_lines = []
        min_sep = 0.01
        for vent, vexit_ref, lvl, src in sorted(candidate_lines, key=lambda r: abs(r[2] - 1.0)):
            if all(abs(vent - s[0]) >= min_sep for s in selected_lines):
                selected_lines.append((vent, vexit_ref, lvl, src))
            if len(selected_lines) >= int(config.PHASE5_REFINEMENT_N_LINES):
                break

        span = float(config.PHASE5_REFINEMENT_VEXIT_SPAN)
        step = max(float(config.PHASE5_REFINEMENT_STEP_V), 1e-4)

        for li, (vent, vexit_ref, lvl, src) in enumerate(selected_lines, start=1):
            vmin = max(bounds_2d[1, 0], vexit_ref - span / 2)
            vmax = min(bounds_2d[1, 1], vexit_ref + span / 2)
            if vmax <= vmin:
                continue

            n_points = int(np.floor((vmax - vmin) / step)) + 1
            if n_points < 3:
                continue
            v_line = np.linspace(vmin, vmax, n_points)

            print(
                f'  Line {li}: V_ENT={vent:.4f} V, source={src}, target n~{lvl:.1f}, '
                f'V_EXIT [{vmin:.4f}, {vmax:.4f}] ({n_points} pts)'
            )

            for vexit in v_line:
                n_val = instr.measure_2d(vent, vexit, V_p_fixed)
                mapper.add_measurement([vent, vexit], n_val, stage='refine_line')
                refinement_points += 1

            mapper.fit()
            refinement_lines.append({
                'V_ent': float(vent),
                'V_exit_ref': float(vexit_ref),
                'target_level': float(lvl),
                'source': src,
                'v_exit_min': float(vmin),
                'v_exit_max': float(vmax),
                'num_points': int(n_points),
            })

        print(f'  Dense refinement complete: {len(refinement_lines)} lines, {refinement_points} points')

    print()
    print(f'Mapping complete: {len(mapper.X_measured)} total measurements')

    results = {
        'phase': 'phase5_pump_map',
        'mapper': mapper,
        'bounds_2d': bounds_2d,
        'bounds_2d_base': bounds_2d_base,
        'bounds_clip_notes': clip_notes,
        'phase2_bounds_used': phase2_bounds_used,
        'replay_bounds_used': replay_bounds_used,
        'V_p_fixed': V_p_fixed,
        'V_ent_center': V_ent_center,
        'V_exit_center': V_exit_center,
        'X_measured': np.array(mapper.X_measured),
        'n_measured': np.array(mapper.y_measured),
        'sample_stage': np.array(mapper.sample_stage),
        'refinement_lines': refinement_lines,
        'refinement_points': int(refinement_points),
        'refinement_line_vents': np.array([row['V_ent'] for row in refinement_lines], dtype=float) if len(refinement_lines) > 0 else np.array([], dtype=float),
    }

    return results


print('Phase 2 and Phase 5 functions defined')

In [ ]:
# Cell 8: Visualization Functions


def _symmetric_limit(values, percentile=99.0, fallback=1e-6):
    arr = np.asarray(values)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return fallback

    lim = np.percentile(np.abs(arr), percentile)
    if not np.isfinite(lim) or lim <= 0:
        lim = np.max(np.abs(arr))
    return max(float(lim), fallback)


def _draw_box(ax, x_bounds, y_bounds, color='w', lw=1.5, ls='--', label=None):
    ax.plot([x_bounds[0], x_bounds[1], x_bounds[1], x_bounds[0], x_bounds[0]],
            [y_bounds[0], y_bounds[0], y_bounds[1], y_bounds[1], y_bounds[0]],
            color=color, lw=lw, ls=ls, label=label)


def plot_phase1_pinchoff(phase1_results, config):
    """Plot gate-wise I-V traces and pinch-off points."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    gates = ['V_ent', 'V_p', 'V_exit']
    titles = ['V_ENT sweep', 'V_P sweep', 'V_EXIT sweep']

    thr_na = phase1_results['pinch_threshold_a'] * 1e9

    for ax, gate, title in zip(axes, gates, titles):
        tr = phase1_results['traces'][gate]
        v = tr['voltage']
        i_na = tr['current_a'] * 1e9
        po = tr['pinch_off_v']

        ax.plot(v, i_na, 'b-', lw=1.8)
        ax.axhline(+thr_na, color='r', ls='--', lw=1)
        ax.axhline(-thr_na, color='r', ls='--', lw=1, label='|I| threshold' if gate == 'V_ent' else None)
        ax.axvline(po, color='k', ls=':', lw=1.5, label='pinch-off' if gate == 'V_ent' else None)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel('Voltage (V)')
        ax.set_ylabel('Current (nA)')
        ax.grid(True, alpha=0.3)

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='upper right')

    plt.suptitle('Phase 1: Pinch-off Search', fontsize=13, fontweight='bold')
    plt.tight_layout()

    config.output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filepath = config.output_dir / f'phase1_pinchoff_{timestamp}.png'
    fig.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {filepath.resolve()}')
    return fig


def plot_phase2_positive_region(phase2_results, config):
    """Review plot for sparse adaptive 3D search result."""
    X = phase2_results['X_measured']
    I_nA = phase2_results['I_measured_nA']
    valid = phase2_results['valid_mask_points']
    vb = phase2_results['valid_bounds_3d']
    best = phase2_results['best_point']

    fig, axes = plt.subplots(1, 3, figsize=(17, 5.3))

    projections = [
        (0, 2, 'V_ENT (V)', 'V_EXIT (V)', 'V_ENT vs V_EXIT'),
        (0, 1, 'V_ENT (V)', 'V_P (V)', 'V_ENT vs V_P'),
        (1, 2, 'V_P (V)', 'V_EXIT (V)', 'V_P vs V_EXIT'),
    ]

    scatters = []
    for ax, (ix, iy, xl, yl, ttl) in zip(axes, projections):
        sc = ax.scatter(X[:, ix], X[:, iy], c=I_nA, cmap='RdYlBu_r', s=28, alpha=0.85, edgecolors='none')
        scatters.append(sc)

        if np.any(valid):
            ax.scatter(X[valid, ix], X[valid, iy], facecolors='none', edgecolors='lime', s=50, linewidths=1.1,
                       label='valid points')

        ax.scatter(best[ix], best[iy], marker='*', s=170, c='gold', edgecolor='k', linewidth=0.9,
                   label='best candidate')

        _draw_box(ax, vb[ix], vb[iy], color='white', lw=1.6, ls='--', label='valid bounds')

        ax.set_xlabel(xl)
        ax.set_ylabel(yl)
        ax.set_title(ttl)
        ax.grid(alpha=0.2)

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        axes[0].legend(loc='best', fontsize=8)

    cbar = fig.colorbar(scatters[-1], ax=axes.ravel().tolist(), shrink=0.9)
    cbar.set_label('Measured current (nA)')

    fig.suptitle(
        f'Phase 2 Attempt {phase2_results["attempt_idx"]}: Sparse Adaptive 3D (0 < I < 4 nA)\n'
        f'Points={len(X)}, Valid={phase2_results["valid_points"]}',
        fontsize=12,
        fontweight='bold'
    )
    plt.tight_layout(rect=[0, 0, 1, 0.92])

    config.output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filepath = config.output_dir / f'phase2_sparse3d_attempt{phase2_results["attempt_idx"]}_{timestamp}.png'
    fig.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {filepath.resolve()}')
    return fig


def plot_phase4_summary(phase4_results, instr, config):
    """Quick summary plot for Phase 4 optimization."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    y_hist = phase4_results['y_hist']
    n_hist = phase4_results['n_hist']
    target_n = phase4_results.get('phase4_target_n', 1.0)

    ax = axes[0]
    ax.plot(y_hist, 'o-', color='steelblue', ms=4, alpha=0.6, label='Cost')
    ax.plot(np.minimum.accumulate(y_hist), 'r-', lw=2, label='Best so far')
    ax.axvline(config.PHASE4_N_INITIAL, color='gray', ls='--', alpha=0.5, label='Init→BO')
    ax.set_xlabel('Iteration', fontsize=12)
    ax.set_ylabel(r'Cost $\log_{10}|n-1|$', fontsize=12)
    ax.set_title('Phase 4: Optimization History', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.hist(n_hist, bins=30, color='steelblue', edgecolor='k', alpha=0.7)
    ax.axvline(target_n, color='red', ls='--', lw=2, label='n=1 (target)')
    ax.axvline(phase4_results['best_n'], color='green', ls='-', lw=2,
               label=f'Best n={phase4_results["best_n"]:.4f}')
    ax.set_xlabel(r'$n$', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title('Phase 4: Distribution of n', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.suptitle(f'Phase 4 Summary [{instr.get_mode_string()}]', fontsize=14, fontweight='bold')
    plt.tight_layout()

    config.output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filepath = config.output_dir / f'phase4_summary_{timestamp}.png'
    fig.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {filepath.resolve()}')
    return fig


def _eq1_n_model(V_exit, alpha, beta, V1, V2, n_scale=1.0, n_offset=0.0):
    """Eq.(1)-style current model converted to normalized current n=I/(ef)."""
    V_exit = np.asarray(V_exit, dtype=float)
    term_left = np.exp(np.clip(-alpha * (V_exit - V1), -80, 80))
    term_right = np.exp(np.clip(beta * (V_exit - V2), -80, 80))
    return n_scale * (1.0 - term_left + term_right) + n_offset


def _fit_eq1_n_model(V_exit, n_values):
    """Fit Eq.(1)-style model to n(V_exit) data using bounded optimization."""
    V = np.asarray(V_exit, dtype=float)
    n = np.asarray(n_values, dtype=float)
    mask = np.isfinite(V) & np.isfinite(n)
    V = V[mask]
    n = n[mask]
    if len(V) < 6:
        return None

    order = np.argsort(V)
    V = V[order]
    n = n[order]

    vmin = float(np.min(V))
    vmax = float(np.max(V))
    vr = max(vmax - vmin, 1e-3)

    q30, q70 = np.quantile(V, [0.30, 0.70])
    starts = [
        np.array([np.log(40.0), np.log(120.0), q30, q70, 1.0, 0.0], dtype=float),
        np.array([np.log(25.0), np.log(220.0), q30 - 0.05 * vr, q70 + 0.05 * vr, 1.0, 0.0], dtype=float),
    ]

    bounds = [
        (np.log(1.0), np.log(3000.0)),   # alpha
        (np.log(1.0), np.log(5000.0)),   # beta
        (vmin - 0.30 * vr, vmax + 0.30 * vr),  # V1
        (vmin - 0.30 * vr, vmax + 0.30 * vr),  # V2
        (0.25, 3.0),                     # n_scale
        (-1.5, 1.5),                     # n_offset
    ]

    def objective(p):
        alpha = float(np.exp(p[0]))
        beta = float(np.exp(p[1]))
        V1 = float(p[2])
        V2 = float(p[3])
        n_scale = float(p[4])
        n_offset = float(p[5])

        pred = _eq1_n_model(V, alpha, beta, V1, V2, n_scale, n_offset)
        resid = pred - n

        # Soft constraints for physically sensible shape
        penalty = 0.0
        if V1 > V2:
            penalty += 50.0 * ((V1 - V2) / vr) ** 2
        plateau_level = n_scale + n_offset
        penalty += 0.05 * ((plateau_level - 1.0) / 0.5) ** 2

        return float(np.mean(resid ** 2) + penalty)

    best = None
    for p0 in starts:
        try:
            res = minimize(objective, p0, method='L-BFGS-B', bounds=bounds)
            if not res.success:
                continue
            score = objective(res.x)
            if (best is None) or (score < best['score']):
                best = {'x': res.x.copy(), 'score': float(score)}
        except Exception:
            continue

    if best is None:
        return None

    p = best['x']
    return {
        'alpha': float(np.exp(p[0])),
        'beta': float(np.exp(p[1])),
        'V1': float(p[2]),
        'V2': float(p[3]),
        'n_scale': float(p[4]),
        'n_offset': float(p[5]),
        'score': float(best['score']),
    }


def plot_phase2_maps(phase2_results, instr, config):
    """6-panel pump map visualization (reused for Phase 5)."""
    fig = plt.figure(figsize=(18, 11))
    gs = gridspec.GridSpec(2, 3, hspace=0.3, wspace=0.3)

    mapper = phase2_results['mapper']
    bounds_2d = phase2_results['bounds_2d']
    V_p_fixed = phase2_results['V_p_fixed']
    X_meas = phase2_results['X_measured']
    n_meas = phase2_results['n_measured']

    res = config.MAPPING_GRID_RESOLUTION
    V_ent_range = np.linspace(bounds_2d[0, 0], bounds_2d[0, 1], res)
    V_exit_range = np.linspace(bounds_2d[1, 0], bounds_2d[1, 1], res)

    V_ENT, V_EXIT, n_gp_mean, n_gp_std = mapper.predict_map(V_ent_range, V_exit_range)

    dn_dV_exit_gp = np.gradient(n_gp_mean, V_exit_range, axis=0)
    dI_dV_exit_gp = config.target_current * dn_dV_exit_gp
    dI_dV_exit_gp_plot = dI_dV_exit_gp * 1e12

    tc_lim = _symmetric_limit(
        dI_dV_exit_gp_plot,
        percentile=config.TRANSCONDUCTANCE_CLIP_PERCENTILE,
        fallback=1e-3
    )

    _, _, n_truth = instr.get_ground_truth_map(V_ent_range, V_exit_range, V_p_fixed)
    mode_str = instr.get_mode_string()

    # Robust n-color limits (avoid hard clipping at n=2)
    n_pool = np.concatenate([n_gp_mean.ravel(), np.asarray(n_meas).ravel()])
    n_pool = n_pool[np.isfinite(n_pool)]
    if n_pool.size == 0:
        n_vmin, n_vmax = 0.0, 2.0
    else:
        n_vmin = float(np.nanpercentile(n_pool, 1))
        n_vmax = float(np.nanpercentile(n_pool, 99))
        n_vmin = min(n_vmin, 0.0, 1.0)
        n_vmax = max(n_vmax, 2.0)
        if n_vmax <= n_vmin + 1e-6:
            n_vmax = n_vmin + 1.0

    contour_levels = [lv for lv in [1.0, 2.0] if (np.nanmin(n_gp_mean) < lv < np.nanmax(n_gp_mean))]

    ax1 = fig.add_subplot(gs[0, 0])
    cm1 = ax1.pcolormesh(
        V_ENT,
        V_EXIT,
        dI_dV_exit_gp_plot,
        cmap='RdBu_r',
        shading='auto',
        vmin=-tc_lim,
        vmax=tc_lim
    )
    fig.colorbar(cm1, ax=ax1, label=r'$dI/dV_{EXIT}$ (pA/V)')
    ax1.scatter(
        phase2_results['V_ent_center'],
        phase2_results['V_exit_center'],
        marker='*', s=150, c='gold', edgecolor='k', linewidth=1.0,
        label='Center'
    )
    ax1.set_xlabel('$V_{ENT}$ (V)')
    ax1.set_ylabel('$V_{EXIT}$ (V)')
    ax1.set_title(f'(a) Pump Map: $dI/dV_{{EXIT}}$ (GP)\n$V_P$={V_p_fixed:.3f}V', fontsize=10)
    ax1.legend(loc='upper right', fontsize=8)

    ax2 = fig.add_subplot(gs[0, 1])
    cm2 = ax2.pcolormesh(V_ENT, V_EXIT, n_gp_mean, cmap='RdYlBu_r', shading='auto', vmin=n_vmin, vmax=n_vmax)
    fig.colorbar(cm2, ax=ax2, label='n (GP mean)')
    ax2.scatter(X_meas[:, 0], X_meas[:, 1], s=8, c='k', alpha=0.35)
    ax2.axhline(phase2_results['V_exit_center'], color='k', ls='--', lw=1, alpha=0.5)
    ax2.axvline(phase2_results['V_ent_center'], color='k', ls='--', lw=1, alpha=0.5)
    if len(contour_levels) > 0:
        cs2 = ax2.contour(V_ENT, V_EXIT, n_gp_mean, levels=contour_levels,
                          colors=['k' if lv == 1.0 else 'white' for lv in contour_levels],
                          linestyles=['--' if lv == 1.0 else '-' for lv in contour_levels],
                          linewidths=1.2)
        fmt2 = {lv: f'n={int(lv)}' for lv in contour_levels}
        ax2.clabel(cs2, inline=True, fmt=fmt2, fontsize=8)
    ax2.set_xlabel('$V_{ENT}$ (V)')
    ax2.set_ylabel('$V_{EXIT}$ (V)')
    ax2.set_title('(b) Pump Map: n (GP mean)', fontsize=10)

    ax3 = fig.add_subplot(gs[0, 2])
    if n_truth is not None:
        n_truth_pool = n_truth[np.isfinite(n_truth)]
        if n_truth_pool.size > 0:
            n3_vmin = min(n_vmin, float(np.nanpercentile(n_truth_pool, 1)))
            n3_vmax = max(n_vmax, float(np.nanpercentile(n_truth_pool, 99)))
        else:
            n3_vmin, n3_vmax = n_vmin, n_vmax

        cm3 = ax3.pcolormesh(
            V_ENT,
            V_EXIT,
            n_truth,
            cmap='RdYlBu_r',
            shading='auto',
            vmin=n3_vmin,
            vmax=n3_vmax
        )
        fig.colorbar(cm3, ax=ax3, label='n (truth)')

        contour_levels_truth = [lv for lv in [1.0, 2.0] if (np.nanmin(n_truth) < lv < np.nanmax(n_truth))]
        if len(contour_levels_truth) > 0:
            cs3 = ax3.contour(V_ENT, V_EXIT, n_truth, levels=contour_levels_truth,
                              colors=['k' if lv == 1.0 else 'white' for lv in contour_levels_truth],
                              linestyles=['--' if lv == 1.0 else '-' for lv in contour_levels_truth],
                              linewidths=1.2)
            fmt3 = {lv: f'n={int(lv)}' for lv in contour_levels_truth}
            ax3.clabel(cs3, inline=True, fmt=fmt3, fontsize=8)

        ax3.set_title('(c) Ground Truth: n (Simulation)', fontsize=10)
    else:
        cm3 = ax3.pcolormesh(V_ENT, V_EXIT, n_gp_std, cmap='hot_r', shading='auto')
        fig.colorbar(cm3, ax=ax3, label='GP std')
        ax3.set_title('(c) GP Uncertainty (Hardware)', fontsize=10)
    ax3.set_xlabel('$V_{ENT}$ (V)')
    ax3.set_ylabel('$V_{EXIT}$ (V)')

    ax4 = fig.add_subplot(gs[1, 0])
    if n_truth is not None:
        delta_gp = np.abs(n_gp_mean - n_truth)
    else:
        delta_gp = np.clip(n_gp_std, 1e-6, None)
    cm4 = ax4.pcolormesh(V_ENT, V_EXIT, delta_gp, cmap='hot_r', shading='auto',
                         norm=LogNorm(vmin=1e-6, vmax=1))
    fig.colorbar(cm4, ax=ax4, label='|Error| / Proxy')
    ax4.scatter(phase2_results['V_ent_center'], phase2_results['V_exit_center'],
                marker='*', s=120, c='cyan', edgecolor='k')
    ax4.set_xlabel('$V_{ENT}$ (V)')
    ax4.set_ylabel('$V_{EXIT}$ (V)')
    ax4.set_title('(d) Error/Uncertainty map (log)', fontsize=10)

    ax5 = fig.add_subplot(gs[1, 1])
    V_exit_curve = np.linspace(bounds_2d[1, 0], bounds_2d[1, 1], 220)

    # Prefer dense-refinement line anchors for visually faithful curves
    curve_vents = []
    if 'refinement_line_vents' in phase2_results and len(phase2_results['refinement_line_vents']) > 0:
        curve_vents = sorted(np.unique(np.asarray(phase2_results['refinement_line_vents'], dtype=float)).tolist())
    if len(curve_vents) == 0:
        curve_vents = []
        for offset in config.CURVE_OFFSETS:
            vent = phase2_results['V_ent_center'] + offset
            if bounds_2d[0, 0] <= vent <= bounds_2d[0, 1]:
                curve_vents.append(float(vent))

    if len(curve_vents) == 0:
        curve_vents = [float(phase2_results['V_ent_center'])]

    colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728', '#9467bd', '#8c564b']
    for i, V_ent_curve in enumerate(curve_vents[:6]):
        c = colors[i % len(colors)]
        n_curve, sigma_curve = mapper.predict_curve(V_exit_curve, V_ent_curve)
        ax5.plot(V_exit_curve, n_curve, color=c, lw=2, label=f'$V_{{ENT}}$={V_ent_curve:.3f}V')
        ax5.fill_between(V_exit_curve, n_curve - sigma_curve, n_curve + sigma_curve, color=c, alpha=0.15)

        # overlay measured local points along the same V_ent line (if enough points)
        tol = max((bounds_2d[0, 1] - bounds_2d[0, 0]) / 80, 0.0015)
        m = np.abs(X_meas[:, 0] - V_ent_curve) <= tol
        if np.any(m):
            order = np.argsort(X_meas[m, 1])
            ax5.plot(X_meas[m, 1][order], n_meas[m][order], 'o', ms=2.8, color=c, alpha=0.45)

    ax5.axhline(1, color='green', ls='--', lw=2, alpha=0.8, label='n=1')
    ax5.axhline(2, color='purple', ls='--', lw=1.6, alpha=0.7, label='n=2')
    ax5.set_xlabel('$V_{EXIT}$ (V)')
    ax5.set_ylabel('n')
    ax5.set_title('(e) Pumping curves at fixed $V_{ENT}$', fontsize=10)
    ax5.grid(alpha=0.25)
    ax5.legend(fontsize=8, loc='best')

    # (f) Fig.3(b)-style uncertainty plot: |(I-ef)/ef| vs V_EXIT (log scale)
    ax6 = fig.add_subplot(gs[1, 2])
    V_ent_best = float(phase2_results['V_ent_center'])

    half_tol = max((bounds_2d[0, 1] - bounds_2d[0, 0]) / 20, 0.002)
    mask = np.abs(X_meas[:, 0] - V_ent_best) <= half_tol

    # Fallback: use nearest-V_ENT points if line points are sparse
    if np.count_nonzero(mask) < 8:
        k = int(min(max(12, len(X_meas) // 6), len(X_meas)))
        nearest_idx = np.argsort(np.abs(X_meas[:, 0] - V_ent_best))[:k]
        mask = np.zeros(len(X_meas), dtype=bool)
        mask[nearest_idx] = True

    X_local = X_meas[mask]
    n_local = n_meas[mask]

    if len(X_local) >= 4:
        order = np.argsort(X_local[:, 1])
        V_exit_local = X_local[order, 1]
        n_local = n_local[order]

        # GP mean evaluated at measured V_EXIT points -> open circles
        n_gp_local, _ = mapper.predict_curve(V_exit_local, V_ent_best)

        err_meas = np.abs(n_local - 1.0)
        err_gp = np.abs(n_gp_local - 1.0)

        y_floor = 1e-10
        y_meas_plot = np.clip(err_meas, y_floor, None)
        y_gp_plot = np.clip(err_gp, y_floor, None)

        ax6.plot(V_exit_local, y_meas_plot, 'o', color='k', ms=4.0, alpha=0.85,
                 label='Measured (solid circle)')
        ax6.plot(V_exit_local, y_gp_plot, 'o', mfc='white', mec='tab:blue', mew=1.2,
                 ms=5.0, alpha=0.9, label='GP mean (open circle)')

        # Red solid line: Eq.(1)-style fit to GP mean curve
        fit = _fit_eq1_n_model(V_exit_local, n_gp_local)
        if fit is not None:
            V_fit = np.linspace(float(np.min(V_exit_local)), float(np.max(V_exit_local)), 280)
            n_fit = _eq1_n_model(
                V_fit,
                fit['alpha'], fit['beta'], fit['V1'], fit['V2'],
                fit['n_scale'], fit['n_offset']
            )
            err_fit = np.clip(np.abs(n_fit - 1.0), y_floor, None)
            ax6.plot(V_fit, err_fit, 'r-', lw=2.0, label='Eq.(1) fit (red)')

            y_all = np.concatenate([y_meas_plot, y_gp_plot, err_fit])
        else:
            y_all = np.concatenate([y_meas_plot, y_gp_plot])

        y_all = y_all[np.isfinite(y_all) & (y_all > 0)]
        if len(y_all) > 0:
            y_min = max(np.min(y_all) * 0.6, y_floor)
            y_max = max(np.percentile(y_all, 99) * 1.8, y_min * 10)
            ax6.set_ylim(y_min, y_max)

    else:
        ax6.text(0.5, 0.5, 'Not enough local points for uncertainty plot',
                 transform=ax6.transAxes, ha='center', va='center', fontsize=9)

    ax6.set_yscale('log')
    ax6.set_xlabel('$V_{EXIT}$ (V)')
    ax6.set_ylabel(r'$\left|\frac{I-ef}{ef}\right|$')
    ax6.set_title(f'(f) Relative pumping error at $V_{{ENT}}$={V_ent_best:.3f}V', fontsize=10)
    ax6.grid(alpha=0.25, which='both')
    ax6.legend(fontsize=8, loc='best')

    plt.suptitle(f'Pump Map Analysis [{mode_str}]', fontsize=15, fontweight='bold')
    plt.tight_layout(rect=[0, 0.00, 1, 0.96])

    config.output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filepath = config.output_dir / f'phase5_pump_maps_{timestamp}.png'
    fig.savefig(filepath, dpi=170, bbox_inches='tight')
    plt.show()
    print(f'Saved: {filepath.resolve()}')
    return fig


print('Visualization functions defined')


In [ ]:
# Cell 9: Save Data Functions


def save_all_data_v5(
    phase1_results,
    phase2_attempts,
    phase2_selected_attempt,
    phase3_decision,
    config,
    phase4_results=None,
    phase5_results=None,
    phase5_decision=None,
):
    """
    Save all available outputs from v5b workflow.
    Includes sparse 3D Phase 2 attempt history and selected attempt.
    """
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    config.output_dir.mkdir(parents=True, exist_ok=True)

    saved_paths = []

    # Phase 1 traces
    for gate, tr in phase1_results['traces'].items():
        df_gate = pd.DataFrame({
            'gate': gate,
            'voltage_V': tr['voltage'],
            'current_A': tr['current_a'],
            'current_nA': tr['current_a'] * 1e9,
            'abs_current_A': tr['abs_current_a'],
        })
        p = config.output_dir / f'phase1_pinchoff_{gate}_{timestamp}.csv'
        df_gate.to_csv(p, index=False)
        saved_paths.append(p)
        print(f'Saved: {p}')

    # Phase 2 attempts (sparse 3D points)
    for attempt in phase2_attempts:
        X = attempt['X_measured']
        valid = attempt['valid_mask_points'].astype(int)
        df_attempt = pd.DataFrame({
            'attempt_idx': int(attempt['attempt_idx']),
            'point_idx': np.arange(len(X)),
            'sample_stage': attempt['sample_stage'],
            'V_ENT': X[:, 0],
            'V_P': X[:, 1],
            'V_EXIT': X[:, 2],
            'I_A': attempt['I_measured_a'],
            'I_nA': attempt['I_measured_nA'],
            'n': attempt['n_measured'],
            'is_valid': valid,
            'adaptive_score': attempt['adaptive_score'],
        })

        p = config.output_dir / f'phase2_attempt_{attempt["attempt_idx"]}_{timestamp}.csv'
        df_attempt.to_csv(p, index=False)
        saved_paths.append(p)
        print(f'Saved: {p}')

    # Phase 4 optimization history (if completed)
    if phase4_results is not None:
        X4 = phase4_results['X_hist']
        df4 = pd.DataFrame({
            'iteration': np.arange(len(X4)),
            'V_ENT': X4[:, 0],
            'V_P': X4[:, 1],
            'V_EXIT': X4[:, 2],
            'n': phase4_results['n_hist'],
            'cost': phase4_results['y_hist'],
            'n_error': np.abs(phase4_results['n_hist'] - phase4_results.get('phase4_target_n', 1.0)),
        })
        p = config.output_dir / f'phase4_optimization_{timestamp}.csv'
        df4.to_csv(p, index=False)
        saved_paths.append(p)
        print(f'Saved: {p}')

    # Phase 5 mapping data (if completed)
    if phase5_results is not None:
        X5 = phase5_results['X_measured']
        stage5 = phase5_results.get('sample_stage', np.array(['sample'] * len(X5)))
        df5 = pd.DataFrame({
            'point_idx': np.arange(len(X5)),
            'sample_stage': stage5,
            'V_ENT': X5[:, 0],
            'V_EXIT': X5[:, 1],
            'V_P': phase5_results['V_p_fixed'],
            'n': phase5_results['n_measured'],
        })
        p = config.output_dir / f'phase5_mapping_{timestamp}.csv'
        df5.to_csv(p, index=False)
        saved_paths.append(p)
        print(f'Saved: {p}')

    selected_idx = np.nan if phase2_selected_attempt is None else int(phase2_selected_attempt + 1)
    selected_valid_points = np.nan
    selected_bounds = None
    if phase2_selected_attempt is not None and len(phase2_attempts) > 0:
        sel = phase2_attempts[phase2_selected_attempt]
        selected_valid_points = int(sel['valid_points'])
        selected_bounds = sel['valid_bounds_3d']

    summary_rows = [
        ('fixed_v_p_mode', bool(getattr(config, 'USE_FIXED_V_P_MODE', False))),
        ('fixed_v_p_value', float(getattr(config, 'FIXED_V_P_VALUE', np.nan)) if getattr(config, 'USE_FIXED_V_P_MODE', False) else np.nan),
        ('simulation_model_kind', str(getattr(config, 'SIMULATION_MODEL_KIND', 'analytic'))),
        ('simulation_replay_data_path', str(getattr(config, 'SIM_REPLAY_DATA_PATH', ''))),
        ('phase1_gain_A_per_V', phase1_results['gain_a_per_v']),
        ('phase1_pinch_V_ent', phase1_results['pinch_off_voltages']['V_ent']),
        ('phase1_pinch_V_p', phase1_results['pinch_off_voltages']['V_p']),
        ('phase1_pinch_V_exit', phase1_results['pinch_off_voltages']['V_exit']),
        ('phase1_measurements', phase1_results['total_measurements']),
        ('phase2_attempt_count', len(phase2_attempts)),
        ('phase2_selected_attempt', selected_idx),
        ('phase2_selected_valid_points', selected_valid_points),
        ('phase3_decision', phase3_decision),
        ('phase4_completed', phase4_results is not None),
        ('phase5_completed', phase5_results is not None),
        ('phase5_decision', phase5_decision if phase5_decision is not None else ''),
    ]

    if selected_bounds is not None:
        summary_rows.extend([
            ('phase2_valid_V_ent_min', selected_bounds[0, 0]),
            ('phase2_valid_V_ent_max', selected_bounds[0, 1]),
            ('phase2_valid_V_p_min', selected_bounds[1, 0]),
            ('phase2_valid_V_p_max', selected_bounds[1, 1]),
            ('phase2_valid_V_exit_min', selected_bounds[2, 0]),
            ('phase2_valid_V_exit_max', selected_bounds[2, 1]),
        ])

    if phase4_results is not None:
        summary_rows.extend([
            ('phase4_best_V_ent', phase4_results['best_V_ent']),
            ('phase4_best_V_p', phase4_results['best_V_p']),
            ('phase4_best_V_exit', phase4_results['best_V_exit']),
            ('phase4_best_n', phase4_results['best_n']),
            ('phase4_best_cost', phase4_results['best_cost']),
            ('phase4_total_measurements', phase4_results['total_measurements_including_refinement']),
        ])

    if phase5_results is not None:
        summary_rows.extend([
            ('phase5_measurements', len(phase5_results['X_measured'])),
            ('phase5_V_p_fixed', phase5_results['V_p_fixed']),
            ('phase5_V_ent_min', phase5_results['bounds_2d'][0, 0]),
            ('phase5_V_ent_max', phase5_results['bounds_2d'][0, 1]),
            ('phase5_V_exit_min', phase5_results['bounds_2d'][1, 0]),
            ('phase5_V_exit_max', phase5_results['bounds_2d'][1, 1]),
            ('phase5_bounds_clip_notes', ';'.join(phase5_results.get('bounds_clip_notes', []))),
            ('phase5_refinement_points', phase5_results.get('refinement_points', 0)),
            ('phase5_refinement_lines', len(phase5_results.get('refinement_lines', []))),
        ])

    summary = pd.DataFrame(summary_rows, columns=['Parameter', 'Value'])
    p_sum = config.output_dir / f'experiment_summary_v5b_{timestamp}.csv'
    summary.to_csv(p_sum, index=False)
    saved_paths.append(p_sum)
    print(f'Saved: {p_sum}')

    return {
        'saved_paths': saved_paths,
        'summary_df': summary,
    }


print('v5b saving functions defined')

In [ ]:
# Runtime override: replay_2d pre-check preset (65 MHz, fixed V_P)
USE_REPLAY2D_PRECHECK_PRESET = True

if USE_REPLAY2D_PRECHECK_PRESET:
    Config.FORCE_SIMULATION = True
    Config.SIMULATION_MODEL_KIND = 'replay_2d'
    Config.SIM_REPLAY_DATA_PATH = '/Users/namkim/Documents/학습-Quantum_BO_Experiment_Hardware_v4 /P1_I-Vx-Vn_p0x2mVp_m1mVs_65MHz1dBm_MUXon_2025032131547copy.txt'
    Config.SIM_REPLAY_V_P_REF = 0.2
    Config.SIM_REPLAY_V_P_SIGMA = 0.03
    Config.SIM_REPLAY_NOISE_STD_N = 0.0

    # Fixed V_P mode
    Config.USE_FIXED_V_P_MODE = True
    Config.FIXED_V_P_VALUE = 0.2
    Config.FIXED_V_P_PHASE2_RANGE = 0.0

    # Phase 1 settings aligned to replay-data domain
    Config.PINCH_REF_V_ENT = 0.2
    Config.PINCH_REF_V_P = 0.2
    Config.PINCH_REF_V_EXIT = 0.03
    Config.PINCH_SCAN_RANGES = {
        'V_ent': (0.05, 0.30),
        'V_p': (0.14, 0.26),
        'V_exit': (-0.20, 0.148),
    }
    Config.PINCH_SCAN_POINTS = 81
    Config.PINCH_OFF_THRESHOLD_A = 2e-12

    # Replay pre-check: avoid bad center drift from Phase1 pinch fallback
    Config.PHASE2_CENTER_FROM_PHASE1_PINCH = False
    Config.PINCH_REF_V_ENT = 0.2
    Config.PINCH_REF_V_P = 0.2
    Config.PINCH_REF_V_EXIT = 0.03
    Config.PHASE2_RANGE_V_ENT = 0.25
    Config.PHASE2_RANGE_V_EXIT = 0.25

    print('Applied runtime preset: replay_2d pre-check (65 MHz, fixed V_P=0.2 V)')
    Config.print_settings()

# Cell 10: RUN Phase 1 + initial Phase 2

config = Config()
config.output_dir.mkdir(parents=True, exist_ok=True)

instr = InstrumentController(config)
print()
print(f'Mode: {instr.get_mode_string()}')

phase2_attempts = []
phase2_selected_attempt = None
phase3_decision = None
phase5_decision = None
phase4_results = None
phase5_results = None

PROCEED_TO_PHASE4 = False
PROCEED_TO_PHASE5 = False

phase1_results = run_phase1_pinchoff(instr, config)
display_phase1_results(phase1_results, config)
fig_phase1 = plot_phase1_pinchoff(phase1_results, config)

attempt_idx = 1
phase2_result = run_phase2_positive_current_mapping(instr, phase1_results, config, attempt_idx=attempt_idx)
phase2_attempts.append(phase2_result)
fig_phase2 = plot_phase2_positive_region(phase2_result, config)

In [ ]:
# Cell 11: PHASE 3 USER REVIEW (y/n/r)

print('='*70)
print('PHASE 3: REVIEW PHASE 2 RESULT')
print('='*70)

phase3_retry_count = 0

while True:
    latest = phase2_attempts[-1]
    print()
    print('Latest Phase 2 attempt summary:')
    print(f'  Attempt #:                  {latest["attempt_idx"]}')
    print(f'  Gain:                       {latest["gain_a_per_v"]:.0e} A/V')
    print(f'  Total measured points:      {len(latest["X_measured"])}')
    print(f'  Valid points:               {latest["valid_points"]}')
    print(f'  Valid bounds V_ENT:         [{latest["valid_bounds_3d"][0,0]:+.4f}, {latest["valid_bounds_3d"][0,1]:+.4f}] V')
    print(f'  Valid bounds V_P:           [{latest["valid_bounds_3d"][1,0]:+.4f}, {latest["valid_bounds_3d"][1,1]:+.4f}] V')
    print(f'  Valid bounds V_EXIT:        [{latest["valid_bounds_3d"][2,0]:+.4f}, {latest["valid_bounds_3d"][2,1]:+.4f}] V')
    print(f'  Best candidate n:           {latest["best_point_n"]:+.5f}')

    print()
    print('Decision options:')
    print('  y: approve Phase 2, manually switch amp gain dial to 1e-9 A/V, then continue to Phase 4')
    print('  n: stop now and save summary')
    print('  r: re-try Phase 2 with same settings')

    response = input('>>> Phase 3 decision [y/n/r]: ').strip().lower()

    if response in ['y', 'yes']:
        phase3_decision = 'y'
        phase2_selected_attempt = len(phase2_attempts) - 1
        PROCEED_TO_PHASE4 = True

        print()
        print('⚠️ Manual action required:')
        print('  1) Turn current amplifier gain dial to 1e-9 A/V')
        input('  2) After switching, press Enter to continue...')

        instr.set_current_amp_gain(config.GAIN_PHASE4_A_PER_V)
        print('✅ Gain sync complete. Proceeding to Phase 4.')
        break

    if response in ['n', 'no']:
        phase3_decision = 'n'
        PROCEED_TO_PHASE4 = False
        print()
        print('⏹️ Stopping after Phase 3 by user decision.')
        break

    if response in ['r', 'retry', 're-try']:
        phase3_decision = 'r'
        phase3_retry_count += 1
        next_attempt = len(phase2_attempts) + 1
        print()
        print(f'🔁 Re-running Phase 2 (attempt {next_attempt})...')
        phase2_result = run_phase2_positive_current_mapping(instr, phase1_results, config, attempt_idx=next_attempt)
        phase2_attempts.append(phase2_result)
        fig_phase2 = plot_phase2_positive_region(phase2_result, config)
        continue

    print('⚠️ Invalid input. Please enter y, n, or r.')

In [ ]:
# Cell 12: RUN PHASE 4 (Constrained n=1 optimization)

if not PROCEED_TO_PHASE4:
    print('='*70)
    print('⏹️ Phase 4 SKIPPED (Phase 3 decision was n)')
    print('='*70)
else:
    selected_phase2 = phase2_attempts[phase2_selected_attempt]
    print(f'Using Phase 2 attempt #{selected_phase2["attempt_idx"]} for constrained bounds.')

    phase4_results = run_phase4_optimization(instr, phase1_results, selected_phase2, config)
    display_phase4_results(phase4_results, config)
    fig_phase4 = plot_phase4_summary(phase4_results, instr, config)

In [ ]:
# Cell 13: PHASE 5 REVIEW + RUN PUMP MAP

if not PROCEED_TO_PHASE4:
    print('⏹️ Skipped - Phase 4 was not executed')
else:
    print('='*70)
    print('PHASE 5 GATE: REVIEW PHASE 4 RESULT')
    print('='*70)
    print(f'Phase 4 best n:     {phase4_results["best_n"]:.6f}')
    print(f'Phase 4 best |n-1|: {abs(phase4_results["best_n"] - phase4_results.get("phase4_target_n", 1.0)):.3e}')

    while True:
        resp = input('>>> Proceed to Phase 5 pump map? [y/n]: ').strip().lower()
        if resp in ['y', 'yes']:
            phase5_decision = 'y'
            PROCEED_TO_PHASE5 = True
            break
        if resp in ['n', 'no']:
            phase5_decision = 'n'
            PROCEED_TO_PHASE5 = False
            break
        print('⚠️ Invalid input. Please enter y or n.')

    if not PROCEED_TO_PHASE5:
        print()
        print('⏹️ Phase 5 skipped by user decision.')
    else:
        print()
        print('>>> Starting Phase 5 mapping...')
        phase5_results = run_phase5_mapping(instr, phase4_results, config, phase2_selected=selected_phase2)
        print('Generating Phase 5 6-panel pump map...')
        fig_phase5 = plot_phase2_maps(phase5_results, instr, config)

In [ ]:
# Cell 14: Save All Data (v5)

save_bundle = save_all_data_v5(
    phase1_results=phase1_results,
    phase2_attempts=phase2_attempts,
    phase2_selected_attempt=phase2_selected_attempt,
    phase3_decision=phase3_decision,
    config=config,
    phase4_results=phase4_results,
    phase5_results=phase5_results,
    phase5_decision=phase5_decision,
)

print()
print('--- Summary Preview ---')
print(save_bundle['summary_df'])

In [ ]:
# Cell 15: Final Summary & Cleanup

print()
print('='*70)
print('EXPERIMENT COMPLETE - FINAL SUMMARY (v5b)')
print('='*70)
print(f'Mode: {instr.get_mode_string()}')

print()
print('--- Phase 1 ---')
print(f'  Gain:              {phase1_results["gain_a_per_v"]:.0e} A/V')
print(f'  Pinch V_ENT:       {phase1_results["pinch_off_voltages"]["V_ent"]:+.6f} V')
print(f'  Pinch V_P:         {phase1_results["pinch_off_voltages"]["V_p"]:+.6f} V')
print(f'  Pinch V_EXIT:      {phase1_results["pinch_off_voltages"]["V_exit"]:+.6f} V')
print(f'  Measurements:      {phase1_results["total_measurements"]}')

print()
print('--- Phase 2 (Sparse 3D) ---')
print(f'  Attempts:          {len(phase2_attempts)}')
if phase2_selected_attempt is not None:
    selected = phase2_attempts[phase2_selected_attempt]
    print(f'  Selected attempt:  {selected["attempt_idx"]}')
    print(f'  Points measured:   {len(selected["X_measured"])}')
    print(f'  Valid points:      {selected["valid_points"]}')
    print(f'  Valid V_ENT:       [{selected["valid_bounds_3d"][0,0]:+.4f}, {selected["valid_bounds_3d"][0,1]:+.4f}] V')
    print(f'  Valid V_P:         [{selected["valid_bounds_3d"][1,0]:+.4f}, {selected["valid_bounds_3d"][1,1]:+.4f}] V')
    print(f'  Valid V_EXIT:      [{selected["valid_bounds_3d"][2,0]:+.4f}, {selected["valid_bounds_3d"][2,1]:+.4f}] V')
else:
    print('  Selected attempt:  None')

print()
print('--- Phase 3 ---')
print(f'  Decision:          {phase3_decision}')

if phase4_results is not None:
    print()
    print('--- Phase 4 ---')
    print(f'  Gain:              {instr.current_amp_gain:.0e} A/V')
    print(f'  Best n:            {phase4_results["best_n"]:.6f}')
    print(f'  Best |n-1|:        {abs(phase4_results["best_n"] - phase4_results.get("phase4_target_n", 1.0)):.3e}')

if phase5_results is not None:
    print()
    print('--- Phase 5 ---')
    print(f'  Decision:          {phase5_decision}')
    print(f'  Measurements:      {len(phase5_results["X_measured"])}')

print()
print('Output directory:')
print(f'  {config.output_dir.resolve()}')
for f in sorted(config.output_dir.glob('*')):
    print(f'    {f.name}')

print('='*70)

instr.close()